# PROYECTO PTIA 2026-2
## StudyFlow AI: capa de órdenes

**Integrantes:**

Karol Ximena Rodriguez Reyes
Juan David Moreno D'Aleman

---

### ¿Qué vamos a hacer en este notebook?

`Estudiante (texto)` -> `Intérprete (LLM)` -> `Confirmación` -> **`Capa de órdenes`** -> `Validador -> MLP -> Capa de asignación` -> `Redactor (LLM)` -> `Estudiante`

`API (FastAPI)` <-> `Persistencia (SQLite)`

El intérprete ya convierte lo que escribe el estudiante en una **orden** de un menú cerrado, pero una orden por sí sola no cambia nada. Esta capa es la que la **aplica**: recibe el plan vigente del estudiante y una orden ya confirmada, cambia lo que la orden pide (horas, fechas, disponibilidad, límites, sesiones fijadas) y vuelve a pasar **solo lo pendiente** por el validador, el MLP y la capa de asignación.

Si lo pedido ya no cabe en el tiempo disponible, la capa no fuerza nada: abre una **decisión pendiente** y le ofrece al estudiante abrir más tiempo, recortar horas (manteniendo intactas las actividades que elija) o cancelar el cambio.

La capa es **determinista** y no usa el LLM ni la base de datos: devuelve qué guardar y qué mostrar, y la API se encarga de guardarlo (persistencia) y de contarlo (redactor).

Cómo se prueba:

* **Pasos 2 a 5:** casos escritos a mano para cada pieza y cada orden, con el MLP real (`modelos/`).
* **Paso 6:** un ejemplo completo, paso a paso, como lo viviría un estudiante.
* **Paso 7:** planes y secuencias de órdenes al azar, con el reloj avanzando, para comprobar que ningún plan aplicado viola una regla.

Todo el código, a excepción de las pruebas, se mudará a `studyflow/ordenes.py`.

# PASO 0. DEPENDENCIAS

Como el de persistencia, este notebook importa el paquete `studyflow/` en vez de copiar el código de las otras capas, porque las usa todas: modelos, validador, asignación, MLP, intérprete (`Orden`) y redactor (`filtrar`). Necesita el entorno del proyecto (`uv sync`) como kernel y la red entrenada en `modelos/` (la genera el paso 6 de `StudyFlow_MLP.ipynb`).

In [1]:
from studyflow.mlp import PredictorMLP

MLP = PredictorMLP()
print("MLP cargado:", MLP.modelo.count_params(), "parámetros")

MLP cargado: 820 parámetros


# PASO 1. REGLAS Y DECISIONES

## 1.1 Dónde encaja

Quien usa esta capa es la API, en cada mensaje que cambia algo:

| Momento | Qué hace la API |
|---|---|
| Llega un mensaje | carga el plan vigente y la decisión pendiente (si hay) e interpreta el mensaje; la decisión pendiente va en `Contexto.decision` |
| El estudiante confirma | llama a `aplicar(vigente, orden, ahora, MLP, decision)` |
| `aplicada` | guarda el plan nuevo (`vigente`), borra la decisión pendiente y el redactor cuenta el horario |
| `decision` | guarda la decisión y muestra el mensaje con las opciones |
| `pregunta` | la decisión sigue pendiente y se muestra la pregunta |
| `cancelada` | borra la decisión; el plan no cambió |
| `rechazada` | no guarda nada y muestra el porqué; si había una decisión pendiente, sigue pendiente |
| `consulta` | el redactor cuenta la vista pedida (no cambia nada ni pide confirmación) |
| Botón *no la hice* | llama a `marcar_no_hecha` (no pasa por el LLM) |
| Formulario del plan | llama a `crear_plan` con el plan completo (actividades, disponibilidad y límites) |

`ahora` lo pone la API con el reloj del servidor, nunca el estudiante.

## 1.2 Lo común a todas las órdenes

* **B1. Qué se rehace:** las sesiones que ya empezaron son historial y no se tocan; cuentan como hechas. Lo demás se vuelve a calcular con validador -> MLP -> asignación, solo con las horas pendientes (total − hecho − fijado).
* **B2. Todo o nada:** la capa devuelve el plan completo o nada. Si algo no se puede, el plan vigente queda igual.
* **B3. Sesiones fijadas:** lo que el estudiante mueve queda fijado y ninguna orden posterior lo deshace. Solo se libera si deja de caber en el plan (un bloqueo, una fecha límite nueva, un límite más bajo o una actividad quitada), y entonces se avisa.
* **B4. No la hice:** una sesión pasada que no se hizo sale del historial; sus horas vuelven a lo pendiente y se replanifican.
* **B5. Hora actual:** hoy solo se planifica desde la hora actual, redondeada al siguiente bloque de 30 min. Lo ya estudiado hoy cuenta para el máximo diario.
* **Reintento antes de decidir:** la asignación atiende las actividades en el orden de prioridad del MLP, y una actividad prioritaria puede ocupar los primeros días y dejar sin espacio a otra que vence antes, aunque el validador diga que todo cabe. Antes de decirle al estudiante que no alcanza, la capa adelanta en el orden a la que quedó incompleta y reintenta; como último recurso, ordena por fecha límite. El resumen sigue mostrando la prioridad que dio el MLP.

## 1.3 Cada orden

| Orden | Qué hace | Cuándo no se aplica |
|---|---|---|
| `cambiar_horas` | nuevo total; lo hecho y lo fijado se descuentan | si el total es menor que lo ya hecho o fijado |
| `cambiar_fecha_limite` | nueva fecha; las fijadas que queden después se liberan | |
| `cambiar_nivel` | nueva prioridad y/o dificultad: cambian las entradas del MLP | |
| `agregar_actividad` | suma la actividad y replanifica todo lo no fijado | nombre repetido; menos de 3 días con disponibilidad antes de su entrega |
| `quitar_actividad` | quita la actividad y todas sus sesiones, **sin mover las demás** | si es la única actividad |
| `bloquear_dia` | quita la disponibilidad de uno o varios días (completos o un rango de horas) y replanifica | si en ese horario no había tiempo de estudio |
| `abrir_tiempo` | agrega disponibilidad en uno o varios días (se une con la que había) y replanifica | si ese tiempo ya estaba disponible |
| `cambiar_limites` | nuevo máximo diario (o ninguno) y/o duración de sesión; las fijadas que no cumplan se liberan | si no cambia nada, o si la sesión queda más larga que el máximo diario |
| `mover_sesion` | fija las sesiones del día de origen en el destino (a la hora pedida o en el primer hueco donde quepan) y replanifica el resto; si en el destino había otra actividad, esa se desplaza | sin sesiones pendientes en el origen; destino igual o posterior a la entrega; sin tiempo de estudio o sin espacio a esa hora (se dice qué está libre); pasaría el máximo diario |
| `consultar_horario` | devuelve la vista pedida (día y/o actividad) | |
| `responder_decision` | resuelve la decisión pendiente (1.4) | sin decisión pendiente; opción que no se ofreció |

Las fechas ya llegan resueltas por el intérprete (nunca en el pasado ni a más de 90 días).

## 1.4 La decisión pendiente (R1)

Cuando el validador dice que no alcanza el tiempo, o la asignación deja algo incompleto aun con el reintento, la capa no aplica el cambio. Devuelve una `Decision` con el plan propuesto (con el cambio ya hecho), las opciones, cuánto falta y antes de qué fecha.

* **Opciones:** un cambio que el estudiante pidió (horas, fecha, actividad nueva, límites, mover) se puede cancelar. Un bloqueo o una sesión no hecha **ya ocurrieron**, así que solo se ofrece abrir tiempo o recortar.
* **Abrir tiempo:** la capa pregunta qué días y horas puede agregar (o subir el máximo diario). Mientras la decisión esté pendiente, toda orden que cambia algo se aplica **sobre el plan propuesto**, así que *el domingo puedo de 9 a 13* se suma al bloqueo. Si con eso ya cabe, se aplica todo junto; si no, la decisión sigue con lo que falta.
* **Recortar:** se quitan horas a las actividades **no protegidas** que vencen hasta la fecha crítica, empezando por la de menor prioridad del MLP, en bloques de 30 min, hasta que el plan cabe. Una actividad que queda en 0 h se quita. Antes de confirmar, la API muestra el recorte exacto (`proponer_recorte` + `texto_recorte`), y al confirmar se recalcula igual (es determinista).
* **Cancelar:** se descarta el cambio y el plan vigente queda como estaba.
* **Vencimiento:** la decisión la guarda la persistencia y vence a los 30 minutos, igual que la orden pendiente. Si vence, no se aplica nada.

## 1.5 Qué devuelve

`ResultadoOrden` trae el `tipo` (tabla 1.1), un `mensaje` escrito por el código, el plan nuevo (`vigente`) si se aplicó, la `decision` si quedó una pendiente, el `horario` que se le muestra al estudiante y una lista de `avisos` (por ejemplo, una sesión fijada que se liberó o el detalle de un recorte). Ningún texto lo escribe el LLM.

# PASO 2. MODELOS Y UTILIDADES (`ordenes.py`, primera parte)

| Pieza | Para qué |
|---|---|
| `PlanVigente` | lo que la capa recibe y devuelve: plan, horario completo (pasado y futuro) y sesiones fijadas |
| `Decision` | el cambio propuesto que no cabe, sus opciones, cuánto falta y la fecha crítica |
| `Recorte`, `CambioHoras` | la propuesta de recorte: horas antes y después de cada actividad |
| `ResultadoOrden` | lo que devuelve cada función pública de la capa |
| `Predictor` | lo único que la capa le pide al MLP: `predecir(plan, fecha_inicio)` |
| `_restar`, `_sumar`, `_cubre`, `_libres` | operaciones sobre las franjas de un día: bloquear, abrir, comprobar que una sesión cabe y ver qué está libre |
| `_pasada`, `_corte` | qué ya empezó y desde qué minuto se puede planificar hoy |
| `_resumen` | el resumen por actividad recalculado a partir de las sesiones |

In [2]:
"""Capa de aplicación de órdenes: modelos y utilidades. Ver StudyFlow_CapaOrdenes.ipynb, paso 2."""
from datetime import date, datetime, time, timedelta
from math import ceil
from typing import Literal, Protocol

from pydantic import BaseModel, ValidationError

from studyflow.asignacion import EstadoAsignacion, Prediccion, ResultadoAsignacion, ResumenActividad, Sesion, asignar
from studyflow.disponibilidad import BLOQUE_MIN, minutos
from studyflow.interprete import Opcion, Orden
from studyflow.llm import duracion_texto, fecha_texto
from studyflow.models import Actividad, DisponibilidadDia, Franja, PlanEntrada
from studyflow.redactor import filtrar
from studyflow.validador import validar

MEDIO_BLOQUE = BLOQUE_MIN / 60 / 2      # menos de esto pendiente no alcanza para un bloque: se da por cumplido
REALIDAD: list[Opcion] = ["abrir_tiempo", "recortar"]               # un bloqueo o una sesión no hecha ya ocurrieron: no se cancelan
DESEO: list[Opcion] = ["abrir_tiempo", "recortar", "cancelar"]      # un cambio pedido sí se puede cancelar


class Predictor(Protocol):
    """Lo único que la capa necesita del MLP (en producción, `PredictorMLP`)."""

    def predecir(self, plan: PlanEntrada, fecha_inicio: date) -> list[Prediccion]: ...


class PlanVigente(BaseModel):
    """Lo que la capa necesita de un estudiante: su plan, el horario completo (pasado y futuro) y las sesiones que fijó."""

    plan: PlanEntrada
    horario: ResultadoAsignacion
    fijadas: list[Sesion] = []


class Decision(BaseModel):
    """Un cambio que no cabe en el tiempo disponible y espera que el estudiante elija cómo resolverlo."""

    propuesto: PlanVigente          # el plan con el cambio ya aplicado; su horario sigue siendo el vigente
    opciones: list[Opcion]
    faltan_horas: float
    fecha_critica: date


class CambioHoras(BaseModel):
    """Horas totales de una actividad antes y después de un recorte (0 = se quita)."""

    actividad: str
    antes: float
    despues: float


class Recorte(BaseModel):
    """Propuesta de recorte: cuántas horas pierde cada actividad para que el plan quepa."""

    cambios: list[CambioHoras]
    alcanza: bool


class ResultadoOrden(BaseModel):
    """Lo que devuelve la capa: qué pasó, qué guardar y qué mostrar."""

    tipo: Literal["aplicada", "decision", "rechazada", "consulta", "pregunta", "cancelada"]
    mensaje: str
    vigente: PlanVigente | None = None              # aplicada: el plan nuevo que se guarda
    decision: Decision | None = None                # decision / pregunta: la decisión que queda pendiente
    horario: ResultadoAsignacion | None = None      # aplicada: el horario completo; consulta: la vista pedida
    avisos: list[str] = []


# --- Utilidades de tiempo y franjas ---------------------------------------------------------------
def _hora(m: int) -> time:
    """Minutos del día -> hora."""
    return time(m // 60, m % 60)


def _dur(s: Sesion) -> int:
    """Duración de una sesión en minutos."""
    return minutos(s.fin) - minutos(s.inicio)


def _horas(sesiones) -> float:
    """Horas totales de un grupo de sesiones."""
    return sum(map(_dur, sesiones)) / 60


def _h(horas: float) -> str:
    """Horas en texto ("1 h 30 min")."""
    return duracion_texto(round(horas * 60))


def _pasada(s: Sesion, ahora: datetime) -> bool:
    """Una sesión que ya empezó cuenta como hecha y no se replanifica."""
    return datetime.combine(s.dia, s.inicio) < ahora


def _corte(ahora: datetime) -> int:
    """Primer minuto planificable de hoy: la hora actual redondeada al siguiente bloque de 30 min."""
    m = ahora.hour * 60 + ahora.minute + (1 if ahora.second or ahora.microsecond else 0)
    return min(ceil(m / BLOQUE_MIN) * BLOQUE_MIN, 24 * 60)


def _traslapan(a: Sesion, b: Sesion) -> bool:
    """Dos sesiones del mismo día que comparten algún minuto."""
    return a.dia == b.dia and minutos(a.inicio) < minutos(b.fin) and minutos(b.inicio) < minutos(a.fin)


def _restar(franjas: list[Franja], ini: int, fin: int) -> list[Franja]:
    """Quita el rango [ini, fin) de las franjas, partiéndolas si hace falta."""
    out = []
    for f in sorted(franjas, key=lambda f: f.inicio):
        a, b = minutos(f.inicio), minutos(f.fin)
        if fin <= a or b <= ini:
            out.append(f)
            continue
        if a < ini:
            out.append(Franja(inicio=_hora(a), fin=_hora(ini)))
        if fin < b:
            out.append(Franja(inicio=_hora(fin), fin=_hora(b)))
    return out


def _unir(rangos: list[tuple[int, int]]) -> list[list[int]]:
    """Une rangos [a, b) que se tocan o se traslapan."""
    unidos = []
    for a, b in sorted(rangos):
        if unidos and a <= unidos[-1][1]:
            unidos[-1][1] = max(unidos[-1][1], b)
        else:
            unidos.append([a, b])
    return unidos


def _sumar(franjas: list[Franja], ini: int, fin: int) -> list[Franja]:
    """Agrega el rango [ini, fin) a las franjas del día."""
    return [Franja(inicio=_hora(a), fin=_hora(b)) for a, b in _unir([(minutos(f.inicio), minutos(f.fin)) for f in franjas] + [(ini, fin)])]


def _cubre(franjas: list[Franja], ini: int, fin: int) -> bool:
    """Indica si el rango [ini, fin) cabe entero dentro de la disponibilidad del día."""
    return any(a <= ini and fin <= b for a, b in _unir([(minutos(f.inicio), minutos(f.fin)) for f in franjas]))


def _libres(franjas: list[Franja], ocupadas: list[Sesion], piso: int = 0) -> list[Franja]:
    """Franjas del día sin lo ocupado y sin lo anterior a `piso`."""
    libres = _restar(franjas, 0, piso) if piso else list(franjas)
    for s in ocupadas:
        libres = _restar(libres, minutos(s.inicio), minutos(s.fin))
    return libres


def _rangos_texto(franjas: list[Franja]) -> str:
    """Franjas en texto ("18:00 a 21:00 y 22:00 a 23:00")."""
    partes = [f"{f.inicio:%H:%M} a {f.fin:%H:%M}" for f in franjas]
    return ", ".join(partes[:-1]) + " y " + partes[-1] if len(partes) > 1 else partes[0]


def _dias(dia: date, ultimo: date | None) -> list[date]:
    """Días de una orden, del primero al último (inclusivo)."""
    return [dia + timedelta(days=k) for k in range(((ultimo or dia) - dia).days + 1)]


def _con_franjas(plan: PlanEntrada, nuevas: dict[date, list[Franja]]) -> PlanEntrada:
    """Copia del plan con la disponibilidad de algunos días reemplazada (los días sin franjas se quitan)."""
    por_dia = {d.dia: d.franjas for d in plan.disponibilidad} | nuevas
    return plan.model_copy(update={"disponibilidad": [DisponibilidadDia(dia=d, franjas=f) for d, f in sorted(por_dia.items()) if f]})


def _actividad(plan: PlanEntrada, nombre: str | None) -> Actividad | None:
    """La actividad del plan con ese nombre, o None."""
    return next((a for a in plan.actividades if a.nombre == nombre), None)


def _con_actividad(plan: PlanEntrada, nombre: str, **cambios) -> PlanEntrada:
    """Copia del plan con una actividad modificada."""
    return plan.model_copy(update={"actividades": [a.model_copy(update=cambios) if a.nombre == nombre else a for a in plan.actividades]})


def _resumen(plan: PlanEntrada, sesiones: list[Sesion], prioridades: dict[str, float]) -> ResultadoAsignacion:
    """Horario completo: las sesiones ordenadas y el resumen por actividad recalculado a partir de ellas."""
    sesiones = sorted(sesiones, key=lambda s: (s.dia, s.inicio))
    filas = []
    for a in plan.actividades:
        asignadas = _horas([s for s in sesiones if s.actividad == a.nombre])
        faltan = a.horas_estimadas - asignadas
        faltan = 0.0 if faltan < MEDIO_BLOQUE - 1e-9 else round(faltan, 6)
        estado = (EstadoAsignacion.ASIGNADA if faltan == 0 else
                  EstadoAsignacion.NO_ASIGNADA if asignadas == 0 else EstadoAsignacion.PARCIAL)
        filas.append(ResumenActividad(nombre=a.nombre, prioridad=prioridades.get(a.nombre, 0.0), horas_solicitadas=a.horas_estimadas,
                                      horas_asignadas=asignadas, horas_faltantes=faltan, estado=estado))
    return ResultadoAsignacion(sesiones=sesiones, actividades=sorted(filas, key=lambda x: -x.prioridad))


def _rechazo(mensaje: str, decision: Decision | None = None) -> ResultadoOrden:
    """Nada cambia; el estudiante recibe el porqué (si había una decisión pendiente, sigue pendiente)."""
    return ResultadoOrden(tipo="rechazada", mensaje=mensaje, decision=decision)

## 2.1 Datos y pruebas de las utilidades

Fecha de referencia: **lunes 2 de noviembre de 2026 a las 8:00** (`AHORA`), antes de la primera franja del día.

El plan de siempre (`plan_base`): disponibilidad de 18:00 a 21:00 entre semana, sábado de 9:00 a 12:00 y de 14:00 a 17:00, domingo libre; un **parcial de IA** de 8 h para el lunes 9 (prioridad y dificultad altas), un **taller de redes** de 3 h para el jueves 5 y una **lectura** de 2 h para el jueves 12 (baja y baja); sesiones de hasta 90 min.

`revisar` comprueba las reglas que todo plan aplicado debe cumplir: en lo futuro, cada sesión dentro de la disponibilidad, antes de su entrega, con la duración permitida y sin pasar el máximo diario; ninguna sesión traslapada; las fijadas dentro del horario; el resumen coherente con las sesiones y, salvo que se diga lo contrario, todo lo que aún no vence asignado completo.

Cada prueba es una función `test_...`; `correr` las ejecuta y muestra cuáles pasaron. Al migrar, pasan tal cual a pytest.

In [3]:
HOY = date(2026, 11, 2)                 # lunes
AHORA = datetime(2026, 11, 2, 8, 0)     # antes de la primera franja del día


def correr(*pruebas):
    """Ejecuta cada prueba y muestra su nombre si pasó."""
    for prueba in pruebas:
        prueba()
        print("ok", prueba.__name__)


def fr(*pares) -> list[Franja]:
    """Franjas a partir de pares ("HH:MM", "HH:MM")."""
    return [Franja(inicio=i, fin=f) for i, f in pares]


def semana(inicio: date = HOY, n: int = 14) -> list[DisponibilidadDia]:
    """Entre semana de 18:00 a 21:00, sábado de 9:00 a 12:00 y de 14:00 a 17:00, domingo libre."""
    out = []
    for k in range(n):
        d = inicio + timedelta(days=k)
        pares = [] if d.weekday() == 6 else [("09:00", "12:00"), ("14:00", "17:00")] if d.weekday() == 5 else [("18:00", "21:00")]
        if pares:
            out.append(DisponibilidadDia(dia=d, franjas=fr(*pares)))
    return out


def act(nombre: str, dias: int, horas: float, prioridad: str = "media", dificultad: str = "media") -> Actividad:
    """Actividad que vence `dias` días después de HOY."""
    return Actividad(nombre=nombre, prioridad=prioridad, dificultad=dificultad, fecha_limite=HOY + timedelta(days=dias), horas_estimadas=horas)


def plan_base(**cambios) -> PlanEntrada:
    """El plan de siempre: parcial (lunes 9), taller (jueves 5) y lectura (jueves 12), sesiones de hasta 90 min."""
    p = PlanEntrada(actividades=[act("Parcial de IA", 7, 8, "alta", "alta"), act("Taller de redes", 3, 3), act("Lectura", 10, 2, "baja", "baja")],
                    disponibilidad=semana(), duracion_sesion=90)
    return p.model_copy(update=cambios)


def vigente_base() -> PlanVigente:
    """El plan de siempre ya creado a las 8:00 del lunes."""
    r = crear_plan(plan_base(), AHORA, MLP)
    assert r.tipo == "aplicada", r.mensaje
    return r.vigente


def de(v: PlanVigente, nombre: str) -> list[Sesion]:
    """Sesiones de una actividad."""
    return [s for s in v.horario.sesiones if s.actividad == nombre]


def revisar(v: PlanVigente, ahora: datetime, completo: bool = True) -> None:
    """Reglas que todo plan aplicado cumple: restricciones en lo futuro, sin traslapes, fijadas en el horario y resumen coherente."""
    p, r = v.plan, v.horario
    por_nombre = {a.nombre: a for a in p.actividades}
    franjas = {d.dia: d.franjas for d in p.disponibilidad}
    futuras = [s for s in r.sesiones if not _pasada(s, ahora)]
    for s in futuras:
        assert s.dia < por_nombre[s.actividad].fecha_limite, f"después de la entrega: {s}"
        assert BLOQUE_MIN <= _dur(s) <= p.duracion_sesion, f"duración inválida: {s}"
        assert _cubre(franjas.get(s.dia, []), minutos(s.inicio), minutos(s.fin)), f"fuera de disponibilidad: {s}"
    for x, y in zip(r.sesiones, r.sesiones[1:]):
        assert (x.dia, x.inicio) <= (y.dia, y.inicio) and not _traslapan(x, y), f"traslape o desorden: {x} / {y}"
    if p.max_horas_dia is not None:
        for d in {s.dia for s in futuras}:
            assert _horas([s for s in r.sesiones if s.dia == d]) <= p.max_horas_dia + 1e-9, f"supera el máximo diario el {d}"
    assert all(s in r.sesiones and not _pasada(s, ahora) for s in v.fijadas), "fijada perdida o pasada"
    assert [x.nombre for x in r.actividades] and {x.nombre for x in r.actividades} == set(por_nombre)
    for x in r.actividades:
        assert abs(x.horas_asignadas - _horas([s for s in r.sesiones if s.actividad == x.nombre])) < 1e-9
        assert (x.estado == EstadoAsignacion.ASIGNADA) == (x.horas_faltantes == 0)
        if completo and por_nombre[x.nombre].fecha_limite > ahora.date():
            assert x.estado == EstadoAsignacion.ASIGNADA, f"quedó incompleta: {x}"
    assert all(a.prioridad >= b.prioridad for a, b in zip(r.actividades, r.actividades[1:]))


# --- Paso 2: utilidades ----------------------------------------------------------------------------
def test_restar_franjas():
    """Restar un rango corta la franja por un borde, la parte en dos o no la toca."""
    f = fr(("18:00", "21:00"))
    assert _restar(f, minutos(time(19)), minutos(time(20))) == fr(("18:00", "19:00"), ("20:00", "21:00"))
    assert _restar(f, minutos(time(17)), minutos(time(19))) == fr(("19:00", "21:00"))
    assert _restar(f, minutos(time(20)), minutos(time(22))) == fr(("18:00", "20:00"))
    assert _restar(f, minutos(time(10)), minutos(time(12))) == f
    assert _restar(f, minutos(time(18)), minutos(time(21))) == []


def test_sumar_y_cubrir_franjas():
    """Sumar une lo que se toca o se traslapa; cubrir mira la unión de las franjas."""
    f = fr(("18:00", "20:00"))
    assert _sumar(f, minutos(time(20)), minutos(time(22))) == fr(("18:00", "22:00"))
    assert _sumar(f, minutos(time(8)), minutos(time(9))) == fr(("08:00", "09:00"), ("18:00", "20:00"))
    assert _sumar(f, minutos(time(18, 30)), minutos(time(19))) == f
    junto = fr(("18:00", "19:00"), ("19:00", "20:00"))
    assert _cubre(junto, minutos(time(18, 30)), minutos(time(19, 30))) and not _cubre(junto, minutos(time(19, 30)), minutos(time(20, 30)))


def test_libres_y_textos():
    """Lo libre de un día descuenta lo ocupado y lo anterior a la hora actual."""
    ocupada = Sesion(actividad="X", dia=HOY, inicio=time(19), fin=time(19, 30))
    assert _libres(fr(("18:00", "21:00")), [ocupada], piso=minutos(time(18, 30))) == fr(("18:30", "19:00"), ("19:30", "21:00"))
    assert _rangos_texto(fr(("18:30", "19:00"), ("19:30", "21:00"))) == "18:30 a 19:00 y 19:30 a 21:00"
    assert _dias(HOY, HOY + timedelta(days=2)) == [HOY, HOY + timedelta(days=1), HOY + timedelta(days=2)] and _dias(HOY, None) == [HOY]


def test_pasada_y_corte():
    """Una sesión que ya empezó es pasada; hoy se planifica desde el siguiente bloque de 30 min."""
    s = Sesion(actividad="X", dia=HOY, inicio=time(18), fin=time(19))
    assert not _pasada(s, datetime(2026, 11, 2, 18, 0)) and _pasada(s, datetime(2026, 11, 2, 18, 1))
    assert _corte(datetime(2026, 11, 2, 18, 0)) == 18 * 60 and _corte(datetime(2026, 11, 2, 18, 1)) == 18 * 60 + 30
    assert _corte(datetime(2026, 11, 2, 18, 30, 5)) == 19 * 60 and _corte(datetime(2026, 11, 2, 23, 50)) == 24 * 60


def test_resumen():
    """El resumen sale de las sesiones; lo que falta por menos de medio bloque no cuenta como faltante."""
    p = plan_base()
    ss = [Sesion(actividad="Taller de redes", dia=HOY, inicio=time(18), fin=time(19, 30)),
          Sesion(actividad="Lectura", dia=HOY, inicio=time(19, 30), fin=time(21))]
    r = _resumen(p, ss, {"Lectura": 0.2, "Taller de redes": 0.6})
    est = {x.nombre: (x.horas_asignadas, x.horas_faltantes, x.estado.value) for x in r.actividades}
    assert est == {"Taller de redes": (1.5, 1.5, "parcial"), "Lectura": (1.5, 0.5, "parcial"), "Parcial de IA": (0.0, 8.0, "no_asignada")}
    assert [x.nombre for x in r.actividades] == ["Taller de redes", "Lectura", "Parcial de IA"]
    casi = _resumen(p.model_copy(update={"actividades": [act("Lectura", 10, 1.7)]}), ss[1:], {})
    assert casi.actividades[0].estado == EstadoAsignacion.ASIGNADA          # 1 h 42 min pedidas, 1 h 30 min puestas: falta menos de un bloque


correr(test_restar_franjas, test_sumar_y_cubrir_franjas, test_libres_y_textos, test_pasada_y_corte, test_resumen)

ok test_restar_franjas
ok test_sumar_y_cubrir_franjas
ok test_libres_y_textos
ok test_pasada_y_corte
ok test_resumen


# PASO 3. REPLANIFICACIÓN COMÚN (`ordenes.py`, segunda parte)

Todas las órdenes terminan en `_replanificar`, que hace lo de 1.2:

1. `_preparar` separa el historial (lo que ya empezó), las fijadas que siguen siendo válidas y las que hay que liberar, y calcula lo pendiente de cada actividad. Arma dos planes de trabajo con lo pendiente desde ahora: uno para la asignación (con las fijadas y lo de hoy como ocupado) y otro **sin el tiempo ocupado**, para el validador y el MLP, que así ven la capacidad real.
2. El validador revisa lo pendiente en modo replanificación (las actividades nuevas, además, deben tener su ventana mínima de 3 días). Un error que no se arregla con tiempo (ventana mínima, sesión más larga que el máximo diario) se rechaza con un mensaje claro; uno de capacidad abre una decisión.
3. El MLP predice prioridad y reparto de lo pendiente y `_asignar` coloca las sesiones con el reintento de 1.2.
4. El horario completo es historial + fijadas + lo nuevo, con el resumen recalculado.

`crear_plan` es la misma replanificación partiendo de cero: el primer horario, con el plan completo que llega del formulario.

In [4]:
"""Capa de aplicación de órdenes: replanificación común. Ver StudyFlow_CapaOrdenes.ipynb, paso 3."""


class Preparado(BaseModel):
    """Todo lo que la replanificación calcula antes de llamar al validador, al MLP y a la asignación."""

    historial: list[Sesion]          # sesiones que ya empezaron: no se tocan
    fijadas: list[Sesion]            # sesiones futuras que el estudiante fijó y siguen siendo válidas
    soltadas: list[Sesion]           # fijadas que ya no caben en el plan y se liberan
    pendientes: list[Actividad]      # actividades con horas por planificar (horas_estimadas = lo que falta)
    hechas: dict[str, float]         # horas ya cumplidas o fijadas por actividad
    asignable: PlanEntrada | None    # plan para la asignación: lo pendiente, desde ahora
    capacidad: PlanEntrada | None    # el mismo, sin el tiempo ocupado: para el validador y el MLP
    ocupadas: list[Sesion]           # lo que la asignación debe respetar: fijadas e historial de hoy


def _fijadas_validas(fijadas: list[Sesion], plan: PlanEntrada) -> tuple[list[Sesion], list[Sesion]]:
    """Separa las sesiones fijadas que siguen cumpliendo el plan de las que hay que liberar."""
    por_nombre = {a.nombre: a for a in plan.actividades}
    franjas = {d.dia: d.franjas for d in plan.disponibilidad}
    ok, fuera = [], []
    for s in sorted(fijadas, key=lambda s: (s.dia, s.inicio)):
        a = por_nombre.get(s.actividad)
        valida = (a is not None and s.dia < a.fecha_limite and _dur(s) <= plan.duracion_sesion
                  and _cubre(franjas.get(s.dia, []), minutos(s.inicio), minutos(s.fin)) and not any(_traslapan(s, x) for x in ok))
        (ok if valida else fuera).append(s)
    if plan.max_horas_dia is not None:
        for d in sorted({s.dia for s in ok}):
            del_dia = [s for s in ok if s.dia == d]
            if _horas(del_dia) > plan.max_horas_dia + 1e-9:
                fuera += del_dia
                ok = [s for s in ok if s.dia != d]
    return ok, [s for s in fuera if s.actividad in por_nombre]


def _preparar(base: PlanVigente, plan: PlanEntrada, fijadas: list[Sesion], ahora: datetime) -> Preparado:
    """Separa lo hecho de lo pendiente y arma los dos planes de trabajo (B1, B3 y B5)."""
    hoy = ahora.date()
    nombres = {a.nombre for a in plan.actividades}
    historial = [s for s in base.horario.sesiones if _pasada(s, ahora) and s.actividad in nombres]
    fijadas, soltadas = _fijadas_validas([s for s in fijadas if not _pasada(s, ahora)], plan)
    hechas = {a.nombre: _horas([s for s in historial + fijadas if s.actividad == a.nombre]) for a in plan.actividades}
    pendientes = [a.model_copy(update={"horas_estimadas": round(a.horas_estimadas - hechas[a.nombre], 6)}) for a in plan.actividades
                  if a.fecha_limite > hoy and a.horas_estimadas - hechas[a.nombre] >= MEDIO_BLOQUE - 1e-9]
    ocupadas = fijadas + [s for s in historial if s.dia == hoy]
    asignable = capacidad = None
    if pendientes:
        disp, libre = [], []
        for d in plan.disponibilidad:
            if d.dia < hoy:
                continue
            frs = _restar(d.franjas, 0, _corte(ahora)) if d.dia == hoy else list(d.franjas)
            disp.append(DisponibilidadDia(dia=d.dia, franjas=frs))
            libre.append(DisponibilidadDia(dia=d.dia, franjas=_libres(frs, [s for s in ocupadas if s.dia == d.dia])))
        vacio = [DisponibilidadDia(dia=hoy, franjas=[])]
        asignable = plan.model_copy(update={"actividades": pendientes, "disponibilidad": disp or vacio})
        capacidad = plan.model_copy(update={"actividades": pendientes, "disponibilidad": libre or vacio})
    return Preparado(historial=historial, fijadas=fijadas, soltadas=soltadas, pendientes=pendientes, hechas=hechas,
                     asignable=asignable, capacidad=capacidad, ocupadas=ocupadas)


def _texto_error(e) -> str:
    """Mensaje para el estudiante a partir de un error del validador que no se resuelve abriendo tiempo ni recortando."""
    d = e.datos
    if e.codigo == "R2_VENTANA_MINIMA":
        return (f"{d['actividad']} necesita al menos {d['minimo']} días con tiempo de estudio antes de su entrega "
                f"y tiene {d['dias_con_disponibilidad']}. Cambia su fecha límite o abre tiempo antes.")
    if e.codigo == "R3A_SESION_MAYOR_MAXIMO":
        return f"Cada sesión puede durar hasta {d['duracion_sesion_min']} min, más que tu máximo diario ({_h(d['max_horas_dia'])}). Ajusta uno de los dos."
    return e.mensaje + "."


def _texto_decision(d: Decision) -> str:
    """El mensaje que presenta la decisión y sus opciones."""
    nombres = {"abrir_tiempo": "abrir más tiempo (más horas otros días o un máximo diario mayor)",
               "recortar": "recortar horas (dime qué actividades quieres mantener intactas)",
               "cancelar": "cancelar el cambio"}
    opciones = [nombres[o] for o in d.opciones]
    lista = ", ".join(opciones[:-1]) + " o " + opciones[-1] if len(opciones) > 1 else opciones[0]
    return f"Con este cambio no alcanza el tiempo: faltan {_h(d.faltan_horas)} antes del {fecha_texto(d.fecha_critica)}. Puedes {lista}."


def _asignar(p: Preparado, predicciones: list[Prediccion], hoy: date) -> ResultadoAsignacion:
    """Asigna en el orden del MLP; si una actividad queda incompleta, la adelanta y reintenta; al final prueba por fecha límite."""
    nombres = [a.nombre for a in p.pendientes]
    incompletas = lambda r: [x.nombre for x in r.actividades if x.estado != EstadoAsignacion.ASIGNADA]
    adelantadas: list[str] = []
    for _ in range(len(nombres)):
        orden = [Prediccion(prioridad=(x.prioridad + 1) / 2 if n in adelantadas else x.prioridad / 2, perfil=x.perfil)
                 for n, x in zip(nombres, predicciones)]
        r = asignar(p.asignable, orden, hoy, p.ocupadas)
        nuevas = [n for n in incompletas(r) if n not in adelantadas]
        if not nuevas:
            break
        adelantadas += nuevas
    if incompletas(r):
        fechas = sorted(range(len(nombres)), key=lambda i: (p.pendientes[i].fecha_limite, -predicciones[i].prioridad))
        rango = {i: 1 - k / len(nombres) for k, i in enumerate(fechas)}
        por_fecha = asignar(p.asignable, [Prediccion(prioridad=rango[i], perfil=x.perfil) for i, x in enumerate(predicciones)], hoy, p.ocupadas)
        if len(incompletas(por_fecha)) < len(incompletas(r)):
            r = por_fecha
    mlp = dict(zip(nombres, (x.prioridad for x in predicciones)))          # el resumen muestra la prioridad que dio el MLP
    return r.model_copy(update={"actividades": [x.model_copy(update={"prioridad": mlp[x.nombre]}) for x in r.actividades]})


def _replanificar(base: PlanVigente, plan: PlanEntrada, fijadas: list[Sesion], ahora: datetime, mlp: Predictor,
                  opciones: list[Opcion], nuevas: frozenset[str] = frozenset(), avisos: list[str] | None = None) -> ResultadoOrden:
    """Vuelve a calcular lo pendiente con validador -> MLP -> asignación; si no cabe todo, abre una decisión (R1)."""
    try:
        plan = PlanEntrada.model_validate(plan.model_dump())
    except ValidationError:
        return _rechazo("Ese cambio deja el plan sin actividades o sin disponibilidad; crea un plan nuevo en su lugar.")
    hoy = ahora.date()
    p = _preparar(base, plan, fijadas, ahora)
    avisos = list(avisos or []) + [f"La sesión fijada de {s.actividad} del {fecha_texto(s.dia)} a las {s.inicio:%H:%M} "
                                   "se liberó porque ya no cabe en tu plan." for s in p.soltadas]
    prioridades = {x.nombre: x.prioridad for x in base.horario.actividades}

    if p.pendientes:
        v = validar(p.capacidad, hoy, "replanificacion")
        errores = list(v.errores)
        recientes = [a for a in p.pendientes if a.nombre in nuevas]
        if recientes:
            errores += [e for e in validar(p.capacidad.model_copy(update={"actividades": recientes}), hoy, "nuevo").errores
                        if e.codigo == "R2_VENTANA_MINIMA"]
        otros = [e for e in errores if e.codigo != "R1_CAPACIDAD_INSUFICIENTE"]
        if otros:
            return _rechazo(" ".join(dict.fromkeys(_texto_error(e) for e in otros)))
        faltan = fecha = None
        if errores:
            faltan, fecha = errores[0].datos["deficit_horas"], date.fromisoformat(errores[0].datos["fecha_critica"])
        else:
            r = _asignar(p, mlp.predecir(p.capacidad, hoy), hoy)
            incompletas = {x.nombre: x.horas_faltantes for x in r.actividades if x.estado != EstadoAsignacion.ASIGNADA}
            if incompletas:
                faltan = sum(incompletas.values())
                fecha = min(a.fecha_limite for a in p.pendientes if a.nombre in incompletas)
            nuevas_sesiones = r.sesiones
            prioridades |= {x.nombre: x.prioridad for x in r.actividades}
        if faltan is not None:
            d = Decision(propuesto=PlanVigente(plan=plan, horario=base.horario, fijadas=fijadas),
                         opciones=opciones, faltan_horas=faltan, fecha_critica=fecha)
            return ResultadoOrden(tipo="decision", mensaje=_texto_decision(d), decision=d, avisos=avisos)
    else:
        nuevas_sesiones = []

    horario = _resumen(plan, p.historial + p.fijadas + nuevas_sesiones, prioridades)
    vigente = PlanVigente(plan=plan, horario=horario, fijadas=p.fijadas)
    return ResultadoOrden(tipo="aplicada", mensaje="Listo, tu horario quedó actualizado.", vigente=vigente, horario=horario, avisos=avisos)


def crear_plan(plan: PlanEntrada, ahora: datetime, mlp: Predictor) -> ResultadoOrden:
    """Primer horario con el formulario completo (actividades, disponibilidad y límites)."""
    nombres = [a.nombre.casefold() for a in plan.actividades]
    if len(nombres) != len(set(nombres)):
        return _rechazo("Hay actividades con el mismo nombre; cada una debe tener un nombre distinto.")
    vacio = PlanVigente(plan=plan, horario=ResultadoAsignacion(sesiones=[], actividades=[]))
    return _replanificar(vacio, plan, [], ahora, mlp, DESEO, nuevas=frozenset(a.nombre for a in plan.actividades))

## 3.1 Pruebas

In [5]:
# --- Paso 3: replanificación común ----------------------------------------------------------------
def test_crear_plan_factible():
    """El plan de siempre cabe: todo queda asignado antes de su entrega y dentro de la disponibilidad."""
    v = vigente_base()
    revisar(v, AHORA)
    assert v.fijadas == [] and len(v.horario.sesiones) > 0


def test_crear_plan_que_no_cabe_abre_decision():
    """Si lo pedido no cabe, no se crea nada: se abre una decisión con las tres opciones."""
    p = plan_base(actividades=[act("Parcial de IA", 7, 40, "alta", "alta"), act("Lectura", 10, 2)])
    r = crear_plan(p, AHORA, MLP)
    assert r.tipo == "decision" and r.vigente is None and r.decision.opciones == DESEO
    assert r.decision.faltan_horas > 0 and r.decision.fecha_critica == HOY + timedelta(days=7)
    assert "faltan" in r.mensaje and "cancelar el cambio" in r.mensaje


def test_crear_plan_sin_ventana_minima():
    """Una entrega con menos de 3 días con disponibilidad se rechaza: abrir tiempo o recortar no lo arreglan."""
    r = crear_plan(plan_base(actividades=[act("Quiz", 2, 1)]), AHORA, MLP)
    assert r.tipo == "rechazada" and "necesita al menos 3 días" in r.mensaje


def test_crear_plan_con_nombres_repetidos():
    """Dos actividades con el mismo nombre no se aceptan."""
    r = crear_plan(plan_base(actividades=[act("Lectura", 5, 1), act("lectura", 8, 1)]), AHORA, MLP)
    assert r.tipo == "rechazada" and "mismo nombre" in r.mensaje


def test_hoy_se_planifica_desde_ahora():
    """Si se crea el plan a las 19:10, hoy no queda ninguna sesión antes de las 19:30."""
    r = crear_plan(plan_base(), datetime(2026, 11, 2, 19, 10), MLP)
    assert r.tipo == "aplicada"
    assert all(s.inicio >= time(19, 30) for s in r.horario.sesiones if s.dia == HOY)
    revisar(r.vigente, datetime(2026, 11, 2, 19, 10))


correr(test_crear_plan_factible, test_crear_plan_que_no_cabe_abre_decision, test_crear_plan_sin_ventana_minima, test_crear_plan_con_nombres_repetidos, test_hoy_se_planifica_desde_ahora)

ok test_crear_plan_factible
ok test_crear_plan_que_no_cabe_abre_decision
ok test_crear_plan_sin_ventana_minima
ok test_crear_plan_con_nombres_repetidos


ok test_hoy_se_planifica_desde_ahora


# PASO 4. CADA ORDEN (`ordenes.py`, tercera parte)

Una función por orden (tabla 1.3), todas con la misma forma: reciben el plan base (el vigente, o el propuesto si hay una decisión pendiente), la orden, `ahora`, el MLP y las opciones de la decisión pendiente, y terminan en `_replanificar` (salvo `quitar_actividad`, que no mueve nada). `ORDENES` las reúne por acción.

Junto con `crear_plan` (paso 3), las funciones públicas son `aplicar` (toda orden confirmada) y `marcar_no_hecha` (el botón *no la hice*).

In [6]:
"""Capa de aplicación de órdenes: cada orden. Ver StudyFlow_CapaOrdenes.ipynb, paso 4."""


def _sin_cancelar(opciones: list[Opcion] | None) -> list[Opcion]:
    """Opciones para un cambio que ya ocurrió: no se puede cancelar."""
    return [o for o in (opciones or DESEO) if o != "cancelar"]


def _no_existe(nombre: str | None, plan: PlanEntrada) -> ResultadoOrden:
    """Rechazo para una actividad que no está en el plan."""
    return _rechazo(f"No encontré {nombre} en tu plan. Tus actividades: {', '.join(a.nombre for a in plan.actividades)}.")


def _cambiar_horas(base: PlanVigente, o: Orden, ahora: datetime, mlp: Predictor, opciones) -> ResultadoOrden:
    """Nuevo total de horas: lo ya hecho o fijado se descuenta y se replanifica el resto."""
    a = _actividad(base.plan, o.actividad)
    if a is None:
        return _no_existe(o.actividad, base.plan)
    hechas = _preparar(base, base.plan, base.fijadas, ahora).hechas[a.nombre]
    if o.horas < hechas - 1e-9:
        return _rechazo(f"{a.nombre} ya tiene {_h(hechas)} estudiadas o fijadas; el nuevo total no puede ser menor.")
    return _replanificar(base, _con_actividad(base.plan, a.nombre, horas_estimadas=o.horas), base.fijadas, ahora, mlp, opciones or DESEO)


def _cambiar_fecha(base: PlanVigente, o: Orden, ahora: datetime, mlp: Predictor, opciones) -> ResultadoOrden:
    """Nueva fecha límite; las sesiones fijadas que queden después se liberan."""
    a = _actividad(base.plan, o.actividad)
    if a is None:
        return _no_existe(o.actividad, base.plan)
    return _replanificar(base, _con_actividad(base.plan, a.nombre, fecha_limite=o.dia), base.fijadas, ahora, mlp, opciones or DESEO)


def _cambiar_nivel(base: PlanVigente, o: Orden, ahora: datetime, mlp: Predictor, opciones) -> ResultadoOrden:
    """Nueva prioridad y/o dificultad: cambian las entradas del MLP y se replanifica."""
    a = _actividad(base.plan, o.actividad)
    if a is None:
        return _no_existe(o.actividad, base.plan)
    cambios = {k: v for k, v in (("prioridad", o.prioridad), ("dificultad", o.dificultad)) if v is not None}
    return _replanificar(base, _con_actividad(base.plan, a.nombre, **cambios), base.fijadas, ahora, mlp, opciones or DESEO)


def _agregar(base: PlanVigente, o: Orden, ahora: datetime, mlp: Predictor, opciones) -> ResultadoOrden:
    """Suma actividades y replanifica todo lo no fijado (3A); las nuevas deben tener su ventana mínima."""
    existentes = {a.nombre.casefold() for a in base.plan.actividades}
    repetidas = [a.nombre for a in o.actividades if a.nombre.casefold() in existentes]
    if repetidas:
        return _rechazo(f"Ya tienes una actividad llamada {repetidas[0]}. Usa otro nombre.")
    plan = base.plan.model_copy(update={"actividades": base.plan.actividades + o.actividades})
    return _replanificar(base, plan, base.fijadas, ahora, mlp, opciones or DESEO, nuevas=frozenset(a.nombre for a in o.actividades))


def _quitar(base: PlanVigente, o: Orden, ahora: datetime, mlp: Predictor, opciones) -> ResultadoOrden:
    """Quita la actividad y sus sesiones (también las pasadas) sin mover las demás (4A)."""
    a = _actividad(base.plan, o.actividad)
    if a is None:
        return _no_existe(o.actividad, base.plan)
    if len(base.plan.actividades) == 1:
        return _rechazo(f"{a.nombre} es tu única actividad. Si ya terminaste, crea un plan nuevo.")
    plan = base.plan.model_copy(update={"actividades": [x for x in base.plan.actividades if x.nombre != a.nombre]})
    sesiones = [s for s in base.horario.sesiones if s.actividad != a.nombre]
    horario = _resumen(plan, sesiones, {x.nombre: x.prioridad for x in base.horario.actividades})
    fijadas = [s for s in base.fijadas if s.actividad != a.nombre]
    vigente = PlanVigente(plan=plan, horario=horario, fijadas=fijadas)
    if opciones:      # había una decisión pendiente: quitar una actividad puede resolverla, así que se revisa
        return _replanificar(vigente, plan, fijadas, ahora, mlp, opciones)
    return ResultadoOrden(tipo="aplicada", mensaje=f"Listo, quité {a.nombre} y sus sesiones.", vigente=vigente, horario=horario)


def _bloquear(base: PlanVigente, o: Orden, ahora: datetime, mlp: Predictor, opciones) -> ResultadoOrden:
    """Quita disponibilidad (uno o varios días, completos o un rango de horas) y replanifica; no se puede cancelar."""
    por_dia = {d.dia: d.franjas for d in base.plan.disponibilidad}
    nuevas = {}
    for d in _dias(o.dia, o.dia_fin):
        if d in por_dia:
            nuevas[d] = [] if o.hora_inicio is None else _restar(por_dia[d], minutos(o.hora_inicio), minutos(o.hora_fin))
    if all(nuevas[d] == por_dia[d] for d in nuevas):
        return _rechazo("En ese horario no tenías tiempo de estudio, así que no hay nada que bloquear.")
    plan = _con_franjas(base.plan, nuevas)
    if not plan.disponibilidad:
        return _rechazo("Con ese bloqueo no te queda ningún día con tiempo de estudio. Abre tiempo en otros días o crea un plan nuevo.")
    return _replanificar(base, plan, base.fijadas, ahora, mlp, _sin_cancelar(opciones))


def _abrir(base: PlanVigente, o: Orden, ahora: datetime, mlp: Predictor, opciones) -> ResultadoOrden:
    """Agrega disponibilidad en uno o varios días (se une con la que ya había) y replanifica."""
    por_dia = {d.dia: d.franjas for d in base.plan.disponibilidad}
    nuevas = {d: _sumar(por_dia.get(d, []), minutos(o.hora_inicio), minutos(o.hora_fin)) for d in _dias(o.dia, o.dia_fin)}
    if all(nuevas[d] == por_dia.get(d) for d in nuevas):
        return _rechazo("Ya tenías ese tiempo disponible.", decision=None)
    return _replanificar(base, _con_franjas(base.plan, nuevas), base.fijadas, ahora, mlp, opciones or DESEO)


def _limites(base: PlanVigente, o: Orden, ahora: datetime, mlp: Predictor, opciones) -> ResultadoOrden:
    """Nuevo máximo de horas por día y/o duración máxima de sesión; las fijadas que ya no cumplan se liberan."""
    maximo = None if o.sin_maximo_diario else (o.max_horas_dia if o.max_horas_dia is not None else base.plan.max_horas_dia)
    duracion = o.duracion_sesion or base.plan.duracion_sesion
    if (maximo, duracion) == (base.plan.max_horas_dia, base.plan.duracion_sesion):
        return _rechazo("Tus límites ya eran esos.")
    plan = base.plan.model_copy(update={"max_horas_dia": maximo, "duracion_sesion": duracion})
    return _replanificar(base, plan, base.fijadas, ahora, mlp, opciones or DESEO)


def _mover(base: PlanVigente, o: Orden, ahora: datetime, mlp: Predictor, opciones) -> ResultadoOrden:
    """Fija las sesiones del día de origen en el destino (a la hora pedida o en el primer hueco) y replanifica el resto."""
    plan, destino = base.plan, o.dia_destino
    movidas = sorted([s for s in base.horario.sesiones if s.dia == o.dia_origen and not _pasada(s, ahora)
                      and o.actividad in (None, s.actividad)], key=lambda s: s.inicio)
    de = f" de {o.actividad}" if o.actividad else ""
    if not movidas:
        return _rechazo(f"No tienes sesiones{de} pendientes el {fecha_texto(o.dia_origen)}.")
    limite = min(_actividad(plan, s.actividad).fecha_limite for s in movidas)
    if destino >= limite:
        return _rechazo(f"El {fecha_texto(destino)} ya no sirve: la entrega es el {fecha_texto(limite)}. Elige un día anterior.")
    franjas = next((d.franjas for d in plan.disponibilidad if d.dia == destino), [])
    if not franjas:
        return _rechazo(f"El {fecha_texto(destino)} no tienes tiempo de estudio. Ábrelo primero, por ejemplo: ese día puedo de 18:00 a 21:00.")
    resto = [s for s in base.fijadas if s not in movidas]
    ocupadas = [s for s in resto if s.dia == destino] + [s for s in base.horario.sesiones if s.dia == destino and _pasada(s, ahora)]
    total = sum(map(_dur, movidas))
    if plan.max_horas_dia is not None and _horas(ocupadas) + total / 60 > plan.max_horas_dia + 1e-9:
        return _rechazo(f"Con eso el {fecha_texto(destino)} pasaría tu máximo de {_h(plan.max_horas_dia)} de estudio.")
    piso = _corte(ahora) if destino == ahora.date() else 0

    def colocar(inicio: int) -> list[Sesion] | None:
        """Las sesiones movidas una tras otra desde `inicio`, o None si no caben ahí."""
        out, cursor = [], inicio
        for s in movidas:
            nueva = Sesion(actividad=s.actividad, dia=destino, inicio=_hora(cursor), fin=_hora(cursor + _dur(s)))
            if cursor < piso or cursor + _dur(s) >= 24 * 60 or not _cubre(franjas, cursor, cursor + _dur(s)) \
                    or any(_traslapan(nueva, x) for x in ocupadas):
                return None
            out.append(nueva)
            cursor += _dur(s)
        return out

    candidatos = [minutos(o.hora_inicio)] if o.hora_inicio else \
        sorted({m for f in franjas for m in range(minutos(f.inicio), minutos(f.fin), BLOQUE_MIN)})
    fijas = next((x for x in map(colocar, candidatos) if x), None)
    if fijas is None:
        libres = _libres(franjas, ocupadas, piso)
        donde = f" a las {o.hora_inicio:%H:%M}" if o.hora_inicio else ""
        return _rechazo(f"No cabe{'n' if len(movidas) > 1 else ''} {_h(total / 60)}{donde} el {fecha_texto(destino)}. "
                        + (f"Ese día tienes libre de {_rangos_texto(libres)}." if libres else "Ese día ya no tienes tiempo libre."))
    return _replanificar(base, plan, resto + fijas, ahora, mlp, opciones or DESEO)


ORDENES = {"cambiar_horas": _cambiar_horas, "cambiar_fecha_limite": _cambiar_fecha, "cambiar_nivel": _cambiar_nivel,
           "agregar_actividad": _agregar, "quitar_actividad": _quitar, "bloquear_dia": _bloquear, "abrir_tiempo": _abrir,
           "cambiar_limites": _limites, "mover_sesion": _mover}


def aplicar(vigente: PlanVigente | None, orden: Orden, ahora: datetime, mlp: Predictor, decision: Decision | None = None) -> ResultadoOrden:
    """Aplica una orden ya confirmada. Con una decisión pendiente, la orden se suma al cambio propuesto."""
    if orden.accion == "crear_plan":
        return _rechazo("El plan nuevo se crea con el formulario, con tus actividades y tu disponibilidad.")
    if vigente is None:
        return _rechazo("Todavía no tienes un plan. Créalo primero con tus actividades y tu disponibilidad.")
    if orden.accion == "consultar_horario":
        return ResultadoOrden(tipo="consulta", mensaje="", horario=filtrar(vigente.horario, dia=orden.dia, actividad=orden.actividad))
    if orden.accion == "responder_decision":
        return _responder(decision, orden, ahora, mlp)
    base = decision.propuesto if decision else vigente
    resultado = ORDENES[orden.accion](base, orden, ahora, mlp, decision.opciones if decision else None)
    if decision and resultado.tipo == "rechazada":
        resultado.decision = decision          # lo rechazado no cierra la decisión que estaba pendiente
    return resultado


def marcar_no_hecha(vigente: PlanVigente, sesion: Sesion, ahora: datetime, mlp: Predictor, decision: Decision | None = None) -> ResultadoOrden:
    """Botón no la hice (B4): la sesión pasada sale del historial, sus horas vuelven a lo pendiente y se replanifica."""
    base = decision.propuesto if decision else vigente
    if sesion not in base.horario.sesiones or not _pasada(sesion, ahora):
        return _rechazo("Solo puedes marcar como no hecha una sesión de tu horario que ya pasó.", decision)
    horario = base.horario.model_copy(update={"sesiones": [s for s in base.horario.sesiones if s != sesion]})
    base = base.model_copy(update={"horario": horario, "fijadas": [s for s in base.fijadas if s != sesion]})
    return _replanificar(base, base.plan, base.fijadas, ahora, mlp, _sin_cancelar(decision.opciones if decision else None),
                         avisos=[f"Las {_h(_dur(sesion) / 60)} de {sesion.actividad} volvieron a lo pendiente."])

## 4.1 Pruebas

In [7]:
# --- Paso 4: cada orden ----------------------------------------------------------------------------
def test_historial_no_se_toca():
    """Al replanificar días después, lo que ya empezó queda igual y no aparece nada nuevo en el pasado."""
    v = vigente_base()
    luego = datetime(2026, 11, 4, 12, 0)                        # miércoles al mediodía
    antes = [s for s in v.horario.sesiones if _pasada(s, luego)]
    r = aplicar(v, Orden(accion="cambiar_nivel", actividad="Lectura", prioridad="alta"), luego, MLP)
    assert r.tipo == "aplicada"
    assert [s for s in r.horario.sesiones if _pasada(s, luego)] == antes and antes
    revisar(r.vigente, luego)


def test_cambiar_horas():
    """Subir y bajar el total de una actividad cambia sus horas asignadas."""
    v = vigente_base()
    sube = aplicar(v, Orden(accion="cambiar_horas", actividad="Lectura", horas=4), AHORA, MLP)
    baja = aplicar(v, Orden(accion="cambiar_horas", actividad="Lectura", horas=1), AHORA, MLP)
    assert _horas(de(sube.vigente, "Lectura")) == 4 and _horas(de(baja.vigente, "Lectura")) == 1
    revisar(sube.vigente, AHORA)
    revisar(baja.vigente, AHORA)


def test_cambiar_horas_por_debajo_de_lo_hecho():
    """El nuevo total no puede ser menor que lo ya estudiado o fijado."""
    v = vigente_base()
    luego = datetime(2026, 11, 5, 12, 0)
    hecho = _horas([s for s in de(v, "Taller de redes") if _pasada(s, luego)])
    assert hecho > 0
    r = aplicar(v, Orden(accion="cambiar_horas", actividad="Taller de redes", horas=hecho - 0.5), luego, MLP)
    assert r.tipo == "rechazada" and "ya tiene" in r.mensaje


def test_cambiar_fecha_limite():
    """Adelantar la entrega deja todas las sesiones antes de la nueva fecha y libera la fijada que queda después."""
    v = vigente_base()
    sabado = date(2026, 11, 7)
    fijada = aplicar(v, Orden(accion="mover_sesion", actividad="Parcial de IA", dia_origen=de(v, "Parcial de IA")[0].dia,
                              dia_destino=sabado, hora_inicio=time(9)), AHORA, MLP).vigente
    assert any(s.dia == sabado for s in fijada.fijadas)
    r = aplicar(fijada, Orden(accion="cambiar_fecha_limite", actividad="Parcial de IA", dia=sabado), AHORA, MLP)
    assert r.tipo in ("aplicada", "decision")
    if r.tipo == "aplicada":
        assert all(s.dia < sabado for s in de(r.vigente, "Parcial de IA")) and not r.vigente.fijadas
        revisar(r.vigente, AHORA)
    assert any("se liberó" in a for a in r.avisos)


def test_cambiar_nivel():
    """Subir la prioridad y la dificultad de la lectura sube la prioridad que le da el MLP."""
    v = vigente_base()
    r = aplicar(v, Orden(accion="cambiar_nivel", actividad="Lectura", prioridad="alta", dificultad="alta"), AHORA, MLP)
    antes = {x.nombre: x.prioridad for x in v.horario.actividades}["Lectura"]
    despues = {x.nombre: x.prioridad for x in r.horario.actividades}["Lectura"]
    assert despues > antes and _actividad(r.vigente.plan, "Lectura").prioridad.value == "alta"
    revisar(r.vigente, AHORA)


def test_agregar_actividad():
    """Una actividad nueva queda asignada; un nombre repetido o una entrega sin ventana se rechazan."""
    v = vigente_base()
    r = aplicar(v, Orden(accion="agregar_actividad", actividades=[act("Informe", 8, 3)]), AHORA, MLP)
    assert r.tipo == "aplicada" and _horas(de(r.vigente, "Informe")) == 3
    revisar(r.vigente, AHORA)
    assert aplicar(v, Orden(accion="agregar_actividad", actividades=[act("lectura", 8, 3)]), AHORA, MLP).tipo == "rechazada"
    sin_ventana = aplicar(v, Orden(accion="agregar_actividad", actividades=[act("Quiz", 2, 1)]), AHORA, MLP)
    assert sin_ventana.tipo == "rechazada" and "Quiz necesita" in sin_ventana.mensaje


def test_quitar_actividad():
    """Quitar una actividad borra sus sesiones y deja las demás exactamente donde estaban (4A)."""
    v = vigente_base()
    r = aplicar(v, Orden(accion="quitar_actividad", actividad="Lectura"), AHORA, MLP)
    assert r.tipo == "aplicada" and de(r.vigente, "Lectura") == []
    assert r.horario.sesiones == [s for s in v.horario.sesiones if s.actividad != "Lectura"]
    revisar(r.vigente, AHORA)
    solo = crear_plan(plan_base(actividades=[act("Lectura", 10, 2)]), AHORA, MLP).vigente
    assert aplicar(solo, Orden(accion="quitar_actividad", actividad="Lectura"), AHORA, MLP).tipo == "rechazada"


def test_bloquear_dia_completo_y_rango():
    """Un bloqueo de día completo vacía el día; uno por horas solo esa parte."""
    v = vigente_base()
    martes, miercoles = date(2026, 11, 3), date(2026, 11, 4)
    r = aplicar(v, Orden(accion="bloquear_dia", dia=martes), AHORA, MLP)
    assert r.tipo == "aplicada" and all(s.dia != martes for s in r.horario.sesiones)
    revisar(r.vigente, AHORA)
    r = aplicar(v, Orden(accion="bloquear_dia", dia=miercoles, hora_inicio=time(18), hora_fin=time(19, 30)), AHORA, MLP)
    assert r.tipo == "aplicada" and all(s.inicio >= time(19, 30) for s in r.horario.sesiones if s.dia == miercoles)
    assert next(d for d in r.vigente.plan.disponibilidad if d.dia == miercoles).franjas == fr(("19:30", "21:00"))
    revisar(r.vigente, AHORA)


def test_bloquear_varios_dias():
    """Un rango bloquea todos sus días; si nada cabe, la decisión no ofrece cancelar."""
    v = vigente_base()
    r = aplicar(v, Orden(accion="bloquear_dia", dia=date(2026, 11, 3), dia_fin=date(2026, 11, 6)), AHORA, MLP)
    assert r.tipo == "decision" and r.decision.opciones == REALIDAD
    assert all(d.dia not in _dias(date(2026, 11, 3), date(2026, 11, 6)) for d in r.decision.propuesto.plan.disponibilidad)
    r = aplicar(v, Orden(accion="bloquear_dia", dia=date(2026, 11, 10), dia_fin=date(2026, 11, 11)), AHORA, MLP)
    assert r.tipo == "aplicada" and all(s.dia not in (date(2026, 11, 10), date(2026, 11, 11)) for s in r.horario.sesiones)


def test_bloquear_sin_disponibilidad():
    """Bloquear un domingo (sin tiempo de estudio) no cambia nada."""
    r = aplicar(vigente_base(), Orden(accion="bloquear_dia", dia=date(2026, 11, 8)), AHORA, MLP)
    assert r.tipo == "rechazada" and "nada que bloquear" in r.mensaje


def test_abrir_tiempo():
    """Abrir el domingo agrega esa franja; abrir lo que ya había no cambia nada."""
    v = vigente_base()
    domingo = date(2026, 11, 8)
    r = aplicar(v, Orden(accion="abrir_tiempo", dia=domingo, hora_inicio=time(9), hora_fin=time(13)), AHORA, MLP)
    assert r.tipo == "aplicada" and next(d for d in r.vigente.plan.disponibilidad if d.dia == domingo).franjas == fr(("09:00", "13:00"))
    revisar(r.vigente, AHORA)
    r = aplicar(v, Orden(accion="abrir_tiempo", dia=date(2026, 11, 3), hora_inicio=time(20), hora_fin=time(23)), AHORA, MLP)
    assert next(d for d in r.vigente.plan.disponibilidad if d.dia == date(2026, 11, 3)).franjas == fr(("18:00", "23:00"))
    repetido = aplicar(v, Orden(accion="abrir_tiempo", dia=date(2026, 11, 3), hora_inicio=time(18), hora_fin=time(20)), AHORA, MLP)
    assert repetido.tipo == "rechazada"


def test_cambiar_limites():
    """Bajar el máximo diario se respeta; un máximo menor que la sesión se rechaza; también se puede quitar."""
    v = vigente_base()
    r = aplicar(v, Orden(accion="cambiar_limites", max_horas_dia=2.5), AHORA, MLP)
    assert r.tipo == "aplicada" and r.vigente.plan.max_horas_dia == 2.5
    revisar(r.vigente, AHORA)
    malo = aplicar(v, Orden(accion="cambiar_limites", max_horas_dia=1), AHORA, MLP)
    assert malo.tipo == "rechazada" and "máximo diario" in malo.mensaje
    corto = aplicar(v, Orden(accion="cambiar_limites", duracion_sesion=60), AHORA, MLP)
    assert corto.tipo == "aplicada" and all(_dur(s) <= 60 for s in corto.horario.sesiones)
    sin = aplicar(r.vigente, Orden(accion="cambiar_limites", sin_maximo_diario=True), AHORA, MLP)
    assert sin.tipo == "aplicada" and sin.vigente.plan.max_horas_dia is None
    assert aplicar(v, Orden(accion="cambiar_limites", duracion_sesion=90), AHORA, MLP).tipo == "rechazada"


def test_mover_sesion_con_hora():
    """La sesión queda fija en el destino a la hora pedida y el resto se replanifica alrededor."""
    v = vigente_base()
    origen = de(v, "Parcial de IA")[0]
    r = aplicar(v, Orden(accion="mover_sesion", actividad="Parcial de IA", dia_origen=origen.dia, dia_destino=date(2026, 11, 7),
                         hora_inicio=time(9)), AHORA, MLP)
    assert r.tipo == "aplicada"
    movida = [s for s in r.vigente.fijadas if s.actividad == "Parcial de IA"]
    assert movida and movida[0].dia == date(2026, 11, 7) and movida[0].inicio == time(9)
    assert _horas(de(r.vigente, "Parcial de IA")) == 8
    revisar(r.vigente, AHORA)


def test_mover_sesion_sin_hora():
    """Sin hora, la sesión va al primer hueco del destino donde quepa entera."""
    v = vigente_base()
    origen = de(v, "Lectura")[0]
    r = aplicar(v, Orden(accion="mover_sesion", actividad="Lectura", dia_origen=origen.dia, dia_destino=date(2026, 11, 7)), AHORA, MLP)
    assert r.tipo == "aplicada" and r.vigente.fijadas[0].inicio == time(9) and r.vigente.fijadas[0].dia == date(2026, 11, 7)
    revisar(r.vigente, AHORA)


def test_mover_a_un_lugar_ocupado_por_otra_actividad():
    """Si el destino lo ocupaba otra actividad, esa se desplaza al replanificar."""
    v = vigente_base()
    martes = date(2026, 11, 3)
    otra = next(s for s in v.horario.sesiones if s.dia == martes)
    origen = next(s for s in v.horario.sesiones if s.actividad != otra.actividad and s.dia != martes and s.dia < date(2026, 11, 5))
    r = aplicar(v, Orden(accion="mover_sesion", actividad=origen.actividad, dia_origen=origen.dia, dia_destino=martes,
                         hora_inicio=otra.inicio), AHORA, MLP)
    assert r.tipo == "aplicada" and any(s.dia == martes and s.inicio == otra.inicio and s.actividad == origen.actividad for s in r.horario.sesiones)
    revisar(r.vigente, AHORA)


def test_mover_sesion_que_no_cabe():
    """Fuera de la disponibilidad, después de la entrega o sin sesiones en el origen: no cambia nada y se explica."""
    v = vigente_base()
    origen = de(v, "Taller de redes")[0]
    fuera = aplicar(v, Orden(accion="mover_sesion", actividad="Taller de redes", dia_origen=origen.dia, dia_destino=date(2026, 11, 4),
                             hora_inicio=time(20, 30)), AHORA, MLP)
    assert fuera.tipo == "rechazada" and "tienes libre" in fuera.mensaje
    tarde = aplicar(v, Orden(accion="mover_sesion", actividad="Taller de redes", dia_origen=origen.dia, dia_destino=date(2026, 11, 7)), AHORA, MLP)
    assert tarde.tipo == "rechazada" and "la entrega es el jueves 5 de noviembre" in tarde.mensaje
    parcial = de(v, "Parcial de IA")[0]
    domingo = aplicar(v, Orden(accion="mover_sesion", actividad="Parcial de IA", dia_origen=parcial.dia, dia_destino=date(2026, 11, 8)), AHORA, MLP)
    assert domingo.tipo == "rechazada" and "no tienes tiempo de estudio" in domingo.mensaje
    vacio = aplicar(v, Orden(accion="mover_sesion", dia_origen=date(2026, 11, 8), dia_destino=date(2026, 11, 7)), AHORA, MLP)
    assert vacio.tipo == "rechazada" and "No tienes sesiones" in vacio.mensaje


def test_la_fijada_sobrevive_a_otras_ordenes():
    """Una sesión movida sigue en su lugar después de otras órdenes que replanifican."""
    v = vigente_base()
    origen = de(v, "Parcial de IA")[0]
    v = aplicar(v, Orden(accion="mover_sesion", actividad="Parcial de IA", dia_origen=origen.dia, dia_destino=date(2026, 11, 7),
                         hora_inicio=time(14)), AHORA, MLP).vigente
    fijas = list(v.fijadas)
    for o in [Orden(accion="cambiar_nivel", actividad="Lectura", prioridad="alta"), Orden(accion="cambiar_horas", actividad="Taller de redes", horas=4),
              Orden(accion="agregar_actividad", actividades=[act("Informe", 9, 2)])]:
        r = aplicar(v, o, AHORA, MLP)
        assert r.tipo == "aplicada" and all(s in r.horario.sesiones for s in fijas), o.accion
        v = r.vigente
        revisar(v, AHORA)


def test_consultar_y_sin_plan():
    """Consultar filtra el horario sin cambiarlo; sin plan, toda orden pide crearlo primero."""
    v = vigente_base()
    r = aplicar(v, Orden(accion="consultar_horario", dia=date(2026, 11, 3)), AHORA, MLP)
    assert r.tipo == "consulta" and r.vigente is None and all(s.dia == date(2026, 11, 3) for s in r.horario.sesiones)
    r = aplicar(v, Orden(accion="consultar_horario", actividad="Lectura"), AHORA, MLP)
    assert {s.actividad for s in r.horario.sesiones} == {"Lectura"}
    assert aplicar(None, Orden(accion="cambiar_horas", actividad="Lectura", horas=3), AHORA, MLP).tipo == "rechazada"


def test_marcar_no_hecha():
    """Una sesión pasada que no se hizo devuelve sus horas a lo pendiente y se vuelve a ubicar en el futuro."""
    v = vigente_base()
    luego = datetime(2026, 11, 4, 12, 0)
    pasada = next(s for s in v.horario.sesiones if _pasada(s, luego))
    r = marcar_no_hecha(v, pasada, luego, MLP)
    assert r.tipo == "aplicada" and pasada not in r.horario.sesiones
    assert _horas(de(r.vigente, pasada.actividad)) == _horas(de(v, pasada.actividad))
    assert any("volvieron a lo pendiente" in a for a in r.avisos)
    revisar(r.vigente, luego)
    futura = next(s for s in v.horario.sesiones if not _pasada(s, luego))
    assert marcar_no_hecha(v, futura, luego, MLP).tipo == "rechazada"


correr(test_historial_no_se_toca, test_cambiar_horas, test_cambiar_horas_por_debajo_de_lo_hecho, test_cambiar_fecha_limite, test_cambiar_nivel, test_agregar_actividad, test_quitar_actividad, test_bloquear_dia_completo_y_rango, test_bloquear_varios_dias, test_bloquear_sin_disponibilidad, test_abrir_tiempo, test_cambiar_limites, test_mover_sesion_con_hora, test_mover_sesion_sin_hora, test_mover_a_un_lugar_ocupado_por_otra_actividad, test_mover_sesion_que_no_cabe, test_la_fijada_sobrevive_a_otras_ordenes, test_consultar_y_sin_plan, test_marcar_no_hecha)

ok test_historial_no_se_toca


ok test_cambiar_horas


ok test_cambiar_horas_por_debajo_de_lo_hecho


ok test_cambiar_fecha_limite


ok test_cambiar_nivel


ok test_agregar_actividad


... (3 outputs left out)

ok test_abrir_tiempo


ok test_cambiar_limites
ok test_mover_sesion_con_hora


ok test_mover_sesion_sin_hora
ok test_mover_a_un_lugar_ocupado_por_otra_actividad
ok test_mover_sesion_que_no_cabe


ok test_la_fijada_sobrevive_a_otras_ordenes
ok test_consultar_y_sin_plan
ok test_marcar_no_hecha


# PASO 5. DECISIÓN PENDIENTE (`ordenes.py`, cuarta parte)

| Pieza | Para qué |
|---|---|
| `proponer_recorte` | calcula el recorte de 1.4: en cada vuelta replanifica, mira cuánto falta y antes de qué fecha, y quita horas a las no protegidas de menor prioridad del MLP. Si no hay de dónde quitar, devuelve `alcanza = False` |
| `texto_recorte` | el recorte en palabras, para la confirmación |
| `_responder` | aplica la opción elegida: cancelar, preguntar cuánto tiempo abrir o recortar |

In [8]:
"""Capa de aplicación de órdenes: resolución de una decisión pendiente (R1). Ver StudyFlow_CapaOrdenes.ipynb, paso 5."""

MAX_VUELTAS_RECORTE = 40


def _con_recorte(plan: PlanEntrada, recorte: Recorte) -> PlanEntrada:
    """Copia del plan con las horas recortadas; una actividad que queda en 0 h se quita."""
    nuevas = {c.actividad: c.despues for c in recorte.cambios}
    actividades = [a.model_copy(update={"horas_estimadas": nuevas[a.nombre]}) if a.nombre in nuevas else a
                   for a in plan.actividades if nuevas.get(a.nombre, 1) > 0]
    return plan.model_copy(update={"actividades": actividades})


def proponer_recorte(decision: Decision, protegidas: list[str], ahora: datetime, mlp: Predictor) -> Recorte:
    """Cuántas horas quitar a cada actividad no protegida para que el plan quepa, empezando por la de menor prioridad del MLP."""
    base = decision.propuesto
    originales = {a.nombre: a.horas_estimadas for a in base.plan.actividades}
    recorte = Recorte(cambios=[], alcanza=False)
    for _ in range(MAX_VUELTAS_RECORTE):
        plan = _con_recorte(base.plan, recorte)
        r = _replanificar(base, plan, base.fijadas, ahora, mlp, decision.opciones)
        if r.tipo == "aplicada":
            recorte.alcanza = True
            return recorte
        if r.tipo != "decision":
            return recorte
        p = _preparar(base, plan, base.fijadas, ahora)
        prioridad = dict(zip([a.nombre for a in p.pendientes], [x.prioridad for x in mlp.predecir(p.capacidad, ahora.date())]))
        candidatas = sorted([a for a in p.pendientes if a.nombre not in protegidas and a.fecha_limite <= r.decision.fecha_critica],
                            key=lambda a: (prioridad[a.nombre], -a.horas_estimadas))
        if not candidatas:
            return recorte
        falta = ceil(r.decision.faltan_horas / MEDIO_BLOQUE / 2 - 1e-9) * MEDIO_BLOQUE * 2      # en bloques de 30 min
        cambios, quitadas = {c.actividad: c for c in recorte.cambios}, 0
        for a in candidatas:
            if falta <= 1e-9:
                break
            quita = min(a.horas_estimadas, falta)
            queda = a.horas_estimadas - quita
            if queda < MEDIO_BLOQUE:
                quita, queda = a.horas_estimadas, 0.0
            falta -= quita
            total = round(p.hechas[a.nombre] + queda, 6)
            if total < MEDIO_BLOQUE:
                if len(plan.actividades) - quitadas <= 1:
                    return recorte          # no se puede quitar la última actividad del plan
                quitadas, total = quitadas + 1, 0.0
            cambios[a.nombre] = CambioHoras(actividad=a.nombre, antes=originales[a.nombre], despues=total)
        recorte = Recorte(cambios=list(cambios.values()), alcanza=False)
    return recorte


def texto_recorte(recorte: Recorte) -> str:
    """El recorte en palabras, para mostrarlo en la confirmación."""
    partes = [f"{c.actividad} pasa de {_h(c.antes)} a {_h(c.despues)}" if c.despues > 0 else f"{c.actividad} se quita ({_h(c.antes)})"
              for c in recorte.cambios]
    return "; ".join(partes) + "."


def _responder(decision: Decision | None, o: Orden, ahora: datetime, mlp: Predictor) -> ResultadoOrden:
    """Aplica la opción elegida para la decisión pendiente."""
    if decision is None:
        return _rechazo("No hay ninguna decisión pendiente en este momento.")
    if o.opcion not in decision.opciones:
        return _rechazo("Esa opción no está disponible para este cambio.", decision)
    if o.opcion == "cancelar":
        return ResultadoOrden(tipo="cancelada", mensaje="Listo, cancelé el cambio: tu horario sigue como estaba.")
    if o.opcion == "abrir_tiempo":
        maximo = decision.propuesto.plan.max_horas_dia
        subir = f", o sube tu máximo diario (ahora es de {_h(maximo)})" if maximo is not None else ""
        return ResultadoOrden(tipo="pregunta", decision=decision,
                              mensaje=f"Dime qué días y de qué hora a qué hora puedes estudiar de más, por ejemplo: el miércoles puedo de 20:00 a 22:00{subir}. "
                                      f"Faltan {_h(decision.faltan_horas)} antes del {fecha_texto(decision.fecha_critica)}.")
    recorte = proponer_recorte(decision, o.protegidas, ahora, mlp)
    if not recorte.alcanza:
        return _rechazo("Recortando solo las actividades que no protegiste no alcanza. Protege menos actividades o abre más tiempo.", decision)
    base = decision.propuesto
    r = _replanificar(base, _con_recorte(base.plan, recorte), base.fijadas, ahora, mlp, decision.opciones)
    r.avisos = ["Recorté: " + texto_recorte(recorte)] + r.avisos
    return r

## 5.1 Pruebas

El escenario: el plan de siempre con **la semana bloqueada de martes a viernes**. Al taller solo le queda el lunes y al parcial, el lunes y el sábado: faltan horas antes del lunes 9.

In [9]:
# --- Paso 5: decisiones ----------------------------------------------------------------------------
BLOQUEO_SEMANA = Orden(accion="bloquear_dia", dia=date(2026, 11, 3), dia_fin=date(2026, 11, 6))   # de martes a viernes


def decision_bloqueo() -> tuple[PlanVigente, Decision]:
    """El plan de siempre con la semana bloqueada: el parcial ya no cabe."""
    v = vigente_base()
    r = aplicar(v, BLOQUEO_SEMANA, AHORA, MLP)
    assert r.tipo == "decision"
    return v, r.decision


def test_decision_del_bloqueo():
    """La decisión guarda el plan con el bloqueo, el horario vigente intacto, lo que falta y la fecha crítica."""
    v, d = decision_bloqueo()
    assert d.propuesto.horario == v.horario and d.opciones == REALIDAD
    assert d.faltan_horas > 0 and d.fecha_critica <= date(2026, 11, 9)
    assert "cancelar" not in _texto_decision(d) and "recortar horas" in _texto_decision(d)


def test_recortar_protegiendo_el_parcial():
    """Con el parcial protegido, se recorta lo demás que vence antes de la fecha crítica y el plan se aplica."""
    v, d = decision_bloqueo()
    rec = proponer_recorte(d, ["Parcial de IA"], AHORA, MLP)
    assert rec.alcanza and rec.cambios and all(c.actividad != "Parcial de IA" and c.despues < c.antes for c in rec.cambios)
    r = aplicar(v, Orden(accion="responder_decision", opcion="recortar", protegidas=["Parcial de IA"]), AHORA, MLP, d)
    assert r.tipo == "aplicada" and r.avisos[0].startswith("Recorté: ")
    assert _actividad(r.vigente.plan, "Parcial de IA").horas_estimadas == 8 and _horas(de(r.vigente, "Parcial de IA")) == 8
    assert all(s.dia not in _dias(date(2026, 11, 3), date(2026, 11, 6)) for s in r.horario.sesiones)
    revisar(r.vigente, AHORA)


def test_recortar_sin_proteger():
    """Sin protegidas, el recorte empieza por la actividad de menor prioridad del MLP."""
    v, d = decision_bloqueo()
    rec = proponer_recorte(d, [], AHORA, MLP)
    assert rec.alcanza
    r = aplicar(v, Orden(accion="responder_decision", opcion="recortar"), AHORA, MLP, d)
    assert r.tipo == "aplicada"
    revisar(r.vigente, AHORA)


def test_recortar_no_alcanza():
    """Si todo lo que vence antes está protegido, no hay de dónde recortar y la decisión sigue pendiente."""
    v, d = decision_bloqueo()
    r = aplicar(v, Orden(accion="responder_decision", opcion="recortar", protegidas=["Parcial de IA", "Taller de redes"]), AHORA, MLP, d)
    assert r.tipo == "rechazada" and r.decision == d and "no alcanza" in r.mensaje


def test_abrir_tiempo_resuelve_la_decision():
    """Elegir abrir tiempo pregunta cuánto; la orden siguiente se suma al bloqueo y, si alcanza, se aplica todo junto."""
    v, d = decision_bloqueo()
    r = aplicar(v, Orden(accion="responder_decision", opcion="abrir_tiempo"), AHORA, MLP, d)
    assert r.tipo == "pregunta" and r.decision == d and "Dime qué días" in r.mensaje
    poco = aplicar(v, Orden(accion="abrir_tiempo", dia=date(2026, 11, 8), hora_inicio=time(9), hora_fin=time(9, 30)), AHORA, MLP, d)
    assert poco.tipo == "decision" and poco.decision.faltan_horas < d.faltan_horas                 # sigue faltando, pero menos
    r = aplicar(v, Orden(accion="abrir_tiempo", dia=date(2026, 11, 8), hora_inicio=time(9), hora_fin=time(17)), AHORA, MLP, poco.decision)
    assert r.tipo == "aplicada" and any(s.dia == date(2026, 11, 8) for s in r.horario.sesiones)
    assert all(s.dia not in _dias(date(2026, 11, 3), date(2026, 11, 6)) for s in r.horario.sesiones)   # el bloqueo sigue
    revisar(r.vigente, AHORA)


def test_cancelar_solo_un_cambio_pedido():
    """Un bloqueo no se cancela; un cambio de horas que no cabe sí, y el horario queda como estaba."""
    v, d = decision_bloqueo()
    assert aplicar(v, Orden(accion="responder_decision", opcion="cancelar"), AHORA, MLP, d).tipo == "rechazada"
    r = aplicar(v, Orden(accion="cambiar_horas", actividad="Parcial de IA", horas=40), AHORA, MLP)
    assert r.tipo == "decision" and r.decision.opciones == DESEO
    c = aplicar(v, Orden(accion="responder_decision", opcion="cancelar"), AHORA, MLP, r.decision)
    assert c.tipo == "cancelada" and c.vigente is None and "sigue como estaba" in c.mensaje


def test_bloqueo_sobre_una_decision_quita_cancelar():
    """Si llega un bloqueo mientras se decide un cambio pedido, ya no se puede cancelar."""
    v = vigente_base()
    r = aplicar(v, Orden(accion="cambiar_horas", actividad="Parcial de IA", horas=20), AHORA, MLP)
    assert r.tipo == "decision" and "cancelar" in r.decision.opciones
    r2 = aplicar(v, Orden(accion="bloquear_dia", dia=date(2026, 11, 10)), AHORA, MLP, r.decision)
    assert r2.tipo == "decision" and r2.decision.opciones == REALIDAD


def test_sin_decision_y_rechazo_que_la_conserva():
    """Responder sin decisión pendiente no hace nada; un rechazo mientras hay una decisión la mantiene."""
    v, d = decision_bloqueo()
    assert aplicar(v, Orden(accion="responder_decision", opcion="recortar"), AHORA, MLP).tipo == "rechazada"
    r = aplicar(v, Orden(accion="cambiar_horas", actividad="Cálculo", horas=3), AHORA, MLP, d)
    assert r.tipo == "rechazada" and r.decision == d


def test_texto_recorte():
    """El recorte se describe con las horas antes y después."""
    rec = Recorte(cambios=[CambioHoras(actividad="Lectura", antes=2, despues=0.5), CambioHoras(actividad="Taller", antes=3, despues=0)], alcanza=True)
    assert texto_recorte(rec) == "Lectura pasa de 2 h a 30 min; Taller se quita (3 h)."


correr(test_decision_del_bloqueo, test_recortar_protegiendo_el_parcial, test_recortar_sin_proteger, test_recortar_no_alcanza, test_abrir_tiempo_resuelve_la_decision, test_cancelar_solo_un_cambio_pedido, test_bloqueo_sobre_una_decision_quita_cancelar, test_sin_decision_y_rechazo_que_la_conserva, test_texto_recorte)

ok test_decision_del_bloqueo


ok test_recortar_protegiendo_el_parcial


ok test_recortar_sin_proteger
ok test_recortar_no_alcanza


ok test_abrir_tiempo_resuelve_la_decision
ok test_cancelar_solo_un_cambio_pedido
ok test_bloqueo_sobre_una_decision_quita_cancelar
ok test_sin_decision_y_rechazo_que_la_conserva
ok test_texto_recorte


# PASO 6. UN EJEMPLO COMPLETO

Un estudiante pasa por varias situaciones seguidas. Las órdenes van escritas a mano (el intérprete ya se probó en su notebook) y los textos que aparecen son los que escribe el código; en la aplicación, el horario final lo contaría el redactor.

In [10]:
import pandas as pd


def tabla(v: PlanVigente) -> pd.DataFrame:
    """El horario por día, con las sesiones fijadas marcadas."""
    return pd.DataFrame([{"Día": fecha_texto(s.dia), "Inicio": f"{s.inicio:%H:%M}", "Fin": f"{s.fin:%H:%M}", "Actividad": s.actividad,
                          "Fijada": "sí" if s in v.fijadas else ""} for s in v.horario.sesiones])


def resumen(v: PlanVigente) -> pd.DataFrame:
    """Horas pedidas y asignadas por actividad, con la prioridad del MLP."""
    return pd.DataFrame([{"Actividad": x.nombre, "Prioridad (MLP)": round(x.prioridad, 3), "Pedidas": x.horas_solicitadas,
                          "Asignadas": x.horas_asignadas, "Estado": x.estado.value} for x in v.horario.actividades])


# 1. Lunes 8:00: el estudiante crea su plan con el formulario
v = vigente_base()
print("1. Plan creado")
display(resumen(v))

# 2. De martes a viernes no puedo estudiar
r = aplicar(v, BLOQUEO_SEMANA, AHORA, MLP)
print("2.", r.tipo, "->", r.mensaje)

# 3. Deja intacto el parcial y recorta lo demás: antes de confirmar se muestra el recorte exacto
d = r.decision
print("3. Confirmación: Entendí: recortar horas de las demás actividades, manteniendo intacta Parcial de IA.",
      texto_recorte(proponer_recorte(d, ["Parcial de IA"], AHORA, MLP)), "¿Confirmas?")
r = aplicar(v, Orden(accion="responder_decision", opcion="recortar", protegidas=["Parcial de IA"]), AHORA, MLP, d)
print("  ", r.tipo, "->", r.mensaje, *r.avisos)
v = r.vigente

# 4. Pasa las sesiones del parcial del sábado al domingo: el domingo no hay tiempo de estudio
MOVER = Orden(accion="mover_sesion", actividad="Parcial de IA", dia_origen=date(2026, 11, 7), dia_destino=date(2026, 11, 8))
r = aplicar(v, MOVER, AHORA, MLP)
print("4.", r.tipo, "->", r.mensaje)

# 5. El domingo puedo de 9 a 12 y la misma orden: no cabe todo, y se dice qué está libre
v = aplicar(v, Orden(accion="abrir_tiempo", dia=date(2026, 11, 8), hora_inicio=time(9), hora_fin=time(12)), AHORA, MLP).vigente
r = aplicar(v, MOVER, AHORA, MLP)
print("5.", r.tipo, "->", r.mensaje)

# 6. Mejor de 9 a 2 de la tarde y otra vez la orden de mover
v = aplicar(v, Orden(accion="abrir_tiempo", dia=date(2026, 11, 8), hora_inicio=time(12), hora_fin=time(14)), AHORA, MLP).vigente
r = aplicar(v, MOVER, AHORA, MLP)
print("6.", r.tipo, "->", r.mensaje)
v = r.vigente
revisar(v, AHORA)
display(tabla(v), resumen(v))

1. Plan creado


,Actividad,Prioridad (MLP),Pedidas,Asignadas,Estado
0,Parcial de IA,0.536,8.0,8.0,asignada
1,Taller de redes,0.344,3.0,3.0,asignada
2,Lectura,0.161,2.0,2.0,asignada


2. decision -> Con este cambio no alcanza el tiempo: faltan 2 h antes del lunes 9 de noviembre. Puedes abrir más tiempo (más horas otros días o un máximo diario mayor) o recortar horas (dime qué actividades quieres mantener intactas).
3. Confirmación: Entendí: recortar horas de las demás actividades, manteniendo intacta Parcial de IA. Taller de redes pasa de 3 h a 1 h. ¿Confirmas?


   aplicada -> Listo, tu horario quedó actualizado. Recorté: Taller de redes pasa de 3 h a 1 h.
4. rechazada -> El domingo 8 de noviembre no tienes tiempo de estudio. Ábrelo primero, por ejemplo: ese día puedo de 18:00 a 21:00.
5. rechazada -> No caben 4 h 30 min el domingo 8 de noviembre. Ese día tienes libre de 09:00 a 12:00.
6. aplicada -> Listo, tu horario quedó actualizado.


,Día,Inicio,Fin,Actividad,Fijada
0,lunes 2 de noviembre,18:00,19:30,Parcial de IA,
1,lunes 2 de noviembre,19:30,20:30,Taller de redes,
2,lunes 2 de noviembre,20:30,21:00,Lectura,
3,sábado 7 de noviembre,09:00,10:30,Parcial de IA,
4,domingo 8 de noviembre,09:00,10:30,Parcial de IA,sí
5,domingo 8 de noviembre,10:30,12:00,Parcial de IA,sí
6,domingo 8 de noviembre,12:00,13:30,Parcial de IA,sí
7,domingo 8 de noviembre,13:30,14:00,Parcial de IA,
8,lunes 9 de noviembre,18:00,18:30,Lectura,
9,miércoles 11 de noviembre,18:00,19:00,Lectura,


,Actividad,Prioridad (MLP),Pedidas,Asignadas,Estado
0,Parcial de IA,0.585,8.0,8.0,asignada
1,Taller de redes,0.345,1.0,1.0,asignada
2,Lectura,0.178,2.0,2.0,asignada


**Observaciones**

1. El plan de siempre cabe completo. El MLP ordena el parcial (0.536), el taller (0.344) y la lectura (0.161).
2. Al bloquear de martes a viernes, al parcial y al taller solo les quedan el lunes 2 y el sábado 7: 9 h para 11 h, así que **faltan 2 h antes del lunes 9**. Como el bloqueo ya ocurrió, la decisión no ofrece cancelar.
3. Con el parcial protegido, el recorte sale del taller, lo único no protegido que vence antes de la fecha crítica: de 3 h a 1 h, justo lo que faltaba. El estudiante ve el recorte exacto antes de confirmar.
4. Mover al domingo se rechaza porque ese día no hay tiempo de estudio, y el mensaje dice cómo abrirlo.
5. Con el domingo abierto de 9:00 a 12:00 no caben las 4 h 30 min del parcial que había el sábado: se rechaza y se dice qué está libre.
6. Con el domingo de 9:00 a 14:00, las sesiones quedan fijadas de 9:00 a 13:30 y el resto se replanifica; el parcial sigue sumando 8 h y su prioridad del MLP cambia (0.585) porque cambió lo pendiente. Como el día de origen queda abierto para todas las actividades (decisión 6A), la replanificación volvió a poner 1 h 30 min del parcial el sábado a las 9:00. Si se prefiere que el origen quede vedado para la actividad movida, hay que cambiar esa decisión.

# PASO 7. PRUEBA CON PLANES Y ÓRDENES ALEATORIOS

Planes al azar desde el lunes 2 (1 a 5 actividades, tres semanas de franjas variadas, sesiones de 60 a 120 min y a veces un máximo diario). Para cada plan que se crea, 8 órdenes al azar de todos los tipos, con el reloj avanzando entre 0 y 26 horas, y de vez en cuando una sesión pasada marcada como no hecha. Cuando aparece una decisión, se resuelve como lo haría un estudiante: recortando (protegiendo una actividad o ninguna), abriendo tiempo o cancelando.

Después de cada orden aplicada se comprueba `revisar` y que **el historial no se perdió** (solo puede salir la sesión marcada como no hecha); después de cada rechazo, que no cambió nada y que la decisión pendiente sigue igual.

In [11]:
import random
from collections import Counter

NIV = ["baja", "media", "alta"]
OPCIONES_FRANJAS = [[("18:00", "21:00")], [("07:00", "09:30"), ("19:00", "22:00")], [("14:00", "18:30")], [("09:00", "12:00"), ("14:00", "17:00")], []]


def plan_aleatorio(rng: random.Random) -> PlanEntrada:
    """Plan al azar desde HOY: 1 a 5 actividades, 3 semanas de franjas variadas, sesión y tope al azar."""
    horizonte = 21
    disp = [DisponibilidadDia(dia=HOY + timedelta(days=k), franjas=fr(*rng.choice(OPCIONES_FRANJAS))) for k in range(horizonte)]
    acts = [act(f"Actividad {i}", rng.randint(4, horizonte - 1), rng.choice([1, 2, 3.5, 5, 8]), rng.choice(NIV), rng.choice(NIV))
            for i in range(rng.randint(1, 5))]
    return PlanEntrada(actividades=acts, disponibilidad=[d for d in disp if d.franjas] or disp[:1], duracion_sesion=rng.choice([60, 90, 120]),
                       max_horas_dia=rng.choice([None, None, 3, 4]))


def orden_aleatoria(rng: random.Random, v: PlanVigente, ahora: datetime, n: int) -> Orden:
    """Una orden al azar que tendría sentido para ese plan (las fechas, como las resolvería el intérprete: desde hoy)."""
    hoy = ahora.date()
    nombres = [a.nombre for a in v.plan.actividades]
    futuras = [s for s in v.horario.sesiones if not _pasada(s, ahora)]
    dia = lambda: hoy + timedelta(days=rng.randint(0, 12))
    hora = lambda: time(rng.choice([7, 9, 14, 18, 19, 20]), rng.choice([0, 30]))
    tipo = rng.choice(["horas", "fecha", "nivel", "agregar", "quitar", "bloquear", "bloquear_horas", "abrir", "limites", "mover", "mover"])
    if tipo == "horas":
        return Orden(accion="cambiar_horas", actividad=rng.choice(nombres), horas=rng.choice([1, 2.5, 4, 6, 10]))
    if tipo == "fecha":
        return Orden(accion="cambiar_fecha_limite", actividad=rng.choice(nombres), dia=hoy + timedelta(days=rng.randint(1, 18)))
    if tipo == "nivel":
        return Orden(accion="cambiar_nivel", actividad=rng.choice(nombres), prioridad=rng.choice(NIV), dificultad=rng.choice(NIV + [None]))
    if tipo == "agregar":
        return Orden(accion="agregar_actividad", actividades=[Actividad(nombre=f"Nueva {n}", prioridad=rng.choice(NIV), dificultad=rng.choice(NIV),
                                                                        fecha_limite=hoy + timedelta(days=rng.randint(3, 15)), horas_estimadas=rng.choice([1, 2, 4]))])
    if tipo == "quitar":
        return Orden(accion="quitar_actividad", actividad=rng.choice(nombres))
    if tipo == "bloquear":
        d = dia()
        return Orden(accion="bloquear_dia", dia=d, dia_fin=d + timedelta(days=rng.choice([0, 0, 1, 3])))
    if tipo == "bloquear_horas":
        h = hora()
        return Orden(accion="bloquear_dia", dia=dia(), hora_inicio=h, hora_fin=_hora(min(minutos(h) + rng.choice([60, 90, 150]), 23 * 60)))
    if tipo == "abrir":
        h = hora()
        d = dia()
        return Orden(accion="abrir_tiempo", dia=d, dia_fin=d + timedelta(days=rng.choice([0, 2])), hora_inicio=h, hora_fin=_hora(min(minutos(h) + rng.choice([60, 120, 180]), 23 * 60)))
    if tipo == "limites":
        return Orden(accion="cambiar_limites", **rng.choice([{"max_horas_dia": rng.choice([2, 3, 5])}, {"sin_maximo_diario": True}, {"duracion_sesion": rng.choice([30, 60, 120])}]))
    s = rng.choice(futuras) if futuras else None
    origen = s.dia if s else dia()
    return Orden(accion="mover_sesion", actividad=s.actividad if s and rng.random() < 0.7 else None, dia_origen=origen, dia_destino=dia(),
                 hora_inicio=hora() if rng.random() < 0.5 else None)


def responder_al_azar(rng: random.Random, v: PlanVigente, d: Decision, ahora: datetime) -> ResultadoOrden:
    """Resuelve una decisión como lo haría un estudiante: recorta (protegiendo algo o no), abre mucho tiempo o cancela."""
    elegida = rng.choice(d.opciones)
    if elegida == "abrir_tiempo":
        return aplicar(v, Orden(accion="abrir_tiempo", dia=ahora.date(), dia_fin=d.fecha_critica - timedelta(days=1),
                                hora_inicio=time(6), hora_fin=time(23)), ahora, MLP, d)
    protegidas = rng.sample([a.nombre for a in d.propuesto.plan.actividades], k=rng.randint(0, 1))
    return aplicar(v, Orden(accion="responder_decision", opcion=elegida, protegidas=protegidas), ahora, MLP, d)


def test_secuencias_aleatorias():
    """Planes y secuencias de órdenes al azar, con el reloj avanzando: ningún plan aplicado viola una regla ni pierde su historial."""
    rng = random.Random(5)
    tipos, creados = Counter(), 0
    for _ in range(60):
        ahora = AHORA + timedelta(hours=rng.randint(0, 12))
        r = crear_plan(plan_aleatorio(rng), ahora, MLP)
        tipos["crear: " + r.tipo] += 1
        if r.tipo != "aplicada":
            continue
        creados += 1
        v, decision = r.vigente, None
        for n in range(8):
            ahora += timedelta(hours=rng.choice([0, 0, 3, 10, 26]))
            if rng.random() < 0.1 and [s for s in v.horario.sesiones if _pasada(s, ahora)]:
                r = marcar_no_hecha(v, rng.choice([s for s in v.horario.sesiones if _pasada(s, ahora)]), ahora, MLP, decision)
            elif decision is not None:
                r = responder_al_azar(rng, v, decision, ahora)
            else:
                o = orden_aleatoria(rng, v, ahora, n)
                r = aplicar(v, o, ahora, MLP, decision)
            tipos[r.tipo] += 1
            if r.tipo == "aplicada":
                clave = lambda s: (s.actividad, s.dia, s.inicio, s.fin)
                antes = {clave(s) for s in v.horario.sesiones if _pasada(s, ahora) and s.actividad in {a.nombre for a in r.vigente.plan.actividades}}
                despues = {clave(s) for s in r.horario.sesiones}
                assert len(antes - despues) <= 1, "se perdió historial"      # solo la sesión marcada como no hecha puede salir
                revisar(r.vigente, ahora)
                v, decision = r.vigente, None
            elif r.tipo in ("decision", "pregunta"):
                decision = r.decision
                assert decision.faltan_horas > 0
            elif r.tipo == "cancelada":
                decision = None
            elif r.tipo == "rechazada":
                assert r.mensaje and r.vigente is None and r.decision == decision
    print(f"{creados} planes creados; resultados: {dict(tipos)}")
    assert creados > 20 and tipos["aplicada"] > 100 and tipos["decision"] > 0 and tipos["rechazada"] > 0


correr(test_secuencias_aleatorias)

58 planes creados; resultados: {'crear: aplicada': 58, 'aplicada': 338, 'rechazada': 115, 'crear: rechazada': 2, 'decision': 10, 'cancelada': 1}
ok test_secuencias_aleatorias


## 7.1 ¿Cuánto ayuda el reintento?

Planes nuevos al azar que el validador acepta: se asignan una vez solo con el orden del MLP y otra con `_asignar`.

In [12]:
def contar_reintentos(n: int = 300, semilla: int = 11) -> Counter:
    """En planes nuevos que el validador acepta, cuántas veces el orden del MLP deja algo incompleto y si el reintento lo resuelve."""
    rng, cuenta = random.Random(semilla), Counter()
    incompletas = lambda r: sum(x.estado != EstadoAsignacion.ASIGNADA for x in r.actividades)
    for _ in range(n):
        plan = plan_aleatorio(rng)
        p = _preparar(PlanVigente(plan=plan, horario=ResultadoAsignacion(sesiones=[], actividades=[])), plan, [], AHORA)
        if not p.pendientes or validar(p.capacidad, HOY, "nuevo").errores:
            continue
        cuenta["planes que caben según el validador"] += 1
        preds = MLP.predecir(p.capacidad, HOY)
        if incompletas(asignar(p.asignable, preds, HOY, p.ocupadas)):
            cuenta["con el orden del MLP queda algo incompleto"] += 1
            cuenta["el reintento lo resuelve" if incompletas(_asignar(p, preds, HOY)) == 0 else "sigue incompleto: se abre una decisión"] += 1
    return cuenta


print(dict(contar_reintentos()))

{'planes que caben según el validador': 283, 'con el orden del MLP queda algo incompleto': 5, 'el reintento lo resuelve': 5}


**Observaciones**

1. De los 60 planes al azar (semilla 5), 58 se crearon y 2 se rechazaron al crearlos. Sobre los 58 se aplicaron 464 órdenes: 338 se aplicaron, 115 se rechazaron, 10 abrieron una decisión y 1 se canceló. Los rechazos son muchos porque las órdenes son al azar (mover a un día sin tiempo, bloquear donde no había disponibilidad, repetir un límite), y cada uno dejó todo como estaba.
2. Ningún plan aplicado violó una regla de `revisar` ni perdió historial, y ningún rechazo cerró la decisión que estaba pendiente.
3. Las decisiones son pocas porque los planes al azar suelen tener holgura; sus caminos (recortar, abrir tiempo, cancelar) están cubiertos por las pruebas del paso 5.
4. **El reintento importa:** de 283 planes que caben según el validador, en 5 (cerca del 2 %) el orden del MLP dejaba una actividad incompleta, y el reintento resolvió los 5. Sin él, esos estudiantes habrían recibido un no alcanza el tiempo falso. El plan de siempre con el martes bloqueado era uno de esos casos.

# PASO 8. MIGRACIÓN A ARCHIVOS `.py`

| Celdas del notebook | Archivo | Imports que se agregan |
|---|---|---|
| Pasos 2, 3, 4 y 5, celdas de `ordenes.py` | `studyflow/ordenes.py` (una sola, en ese orden) | ninguno: la primera celda ya trae sus imports de `studyflow` |
| Pasos 2.1, 3.1, 4.1, 5.1 y 7 | `tests/test_ordenes.py` | `PredictorMLP` como *fixture* del módulo; los ayudantes (`semana`, `act`, `plan_base`, `vigente_base`, `revisar`, ...) quedan en el mismo archivo |

## Puntos para tener en cuenta en las otras capas

* **Persistencia:** la tabla `sesiones` necesita marcar cuáles están **fijadas**, y la decisión pendiente se guarda junto a la orden pendiente (las dos pueden existir a la vez: una decisión espera y el estudiante pide recortar, lo que crea una orden por confirmar). La decisión vence a los 30 minutos, igual que la orden.
* **API:** antes de mostrar la confirmación de `responder_decision` con `recortar`, la API llama a `proponer_recorte` y agrega `texto_recorte` al mensaje, para que el estudiante confirme el recorte exacto. `ahora` sale del reloj del servidor.
* **Redactor:** con `aplicada` cuenta `horario`; con `consulta`, la vista. Los `avisos` y el `mensaje` se muestran tal cual: ya los escribió el código.
* **Front:** el botón *no la hice* va en cada sesión pasada del horario y llama a `marcar_no_hecha` sin pasar por el LLM.